In [ ]:
import pandas as pd
import numpy as np
from hierarchicalforecast.utils import aggregate
import holidays
import matplotlib.pyplot as plt
import ruptures as rpt
from statsmodels.tsa.seasonal import STL


In [ ]:
sales = pd.read_csv("../eabl parsed csv/sales_products_forecast_ready.csv")
sales


In [ ]:
cm_trimmed = pd.read_excel("../EABL dataset/CustomerMaster_clean_forecast.xlsx")
cm_trimmed

In [ ]:
sales.isna().sum()

In [ ]:
cm_trimmed.isna().sum()

In [ ]:
df = sales.merge(
    cm_trimmed,
    left_on="customer_code",
    right_on="DMS Customer Code",
    how="left"
)



In [ ]:
print(df.shape)
print("unmatched customers:", df["DMS Customer Code"].isna().mean() * 100, "%")

df.to_csv("../eabl parsed csv/hierarchical_forecast_dataset.csv", index=False)

In [ ]:

# Null count and percentage for every column, sorted highest first
null_summary = pd.DataFrame({
    "nulls": df.isna().sum(),
    "pct": df.isna().mean() * 100
}).sort_values("nulls", ascending=False)


print(null_summary[null_summary["nulls"] > 0])


In [ ]:
unmatched = df[df["DMS Customer Code"].isna()]
print(unmatched.shape)
print(unmatched[["customer_code"]])


In [ ]:
df = df[df["DMS Customer Code"].notna()]
print(df.shape)


In [ ]:
# ==========================================================================
# STEP R1 - Separate Returnable (packaging) from the EABL drinks
# Returnable = 15 empty bottles / crates / keg containers, counted in PIECES
# (product master: Default_UOM = PC, Sub_Category1 = Bottles/Crates), not drinks.
# FPP2 1.6 Step 1: forecast what the users need (EABL drinks).
# FPP2 3.4: do not mix series in different units in one accuracy measure.
# FPP2 10.1: the hierarchy total must be a meaningful sum.
#
# From here on, df = DRINKS ONLY (Beer, RTD, Spirits), so every later Prep cell
# (D2, cost analysis, hierarchy, zero-fill, F2, classification) uses drinks only.
# Drinks + Returnable TOGETHER stay in ../eabl parsed csv/sales_products_forecast_ready.csv
# (Cleaning output, unchanged) and in `sales` (Prep cell 1).
#
# Where each input comes from:
#   pd   -> Prep cell 0 (imports)
#   df   -> Prep cell 5 (sales + customer master join), cell 9 (unmatched customers removed)
#           Columns used: Category2, Sub_Category, Brand, Product_Code, visit_date, quantity_cases
# Writes ONE new file: ../eabl parsed csv/Y_returnable.csv
# Run it ONCE per session, straight after cell 9 (use Restart -> Run All).
# ==========================================================================

RSPLIT_CAT = "Returnable"     # the Category2 value of the packaging items

# ---- Safety checks: stop if df is not the table this cell expects ----
assert {"Category2", "Sub_Category", "Brand", "Product_Code", "visit_date", "quantity_cases"}.issubset(df.columns), \
    "`df` is not Prep's joined table - run Prep cells 1 to 9 first"
assert df["DMS Customer Code"].notna().all(), "unmatched customers still in df - run Prep cell 9 first"
assert (df["Category2"] == RSPLIT_CAT).any(), \
    "no Returnable rows left in df - this cell already ran. Use Restart -> Run All (do not run it twice)"

# ---- Before the split (used by the checks below) ----
rsplit_rows_before  = len(df)
rsplit_cases_before = df["quantity_cases"].sum()
rsplit_prods_before = df["Product_Code"].nunique()
rsplit_last_week    = pd.to_datetime(df["visit_date"]).dt.to_period("W").dt.start_time.max()   # Monday of the last week (6 Jul 2026)

# ---- The split ----
rsplit_is_ret = df["Category2"] == RSPLIT_CAT       # True for the 15 packaging items
df_returnable = df[rsplit_is_ret].copy()            # packaging only
df            = df[~rsplit_is_ret].copy()           # drinks only - used by every later cell

# ---- Checks: nothing lost, nothing in both ----
assert len(df) + len(df_returnable) == rsplit_rows_before, "rows lost in the split - stop and check"
assert abs(df["quantity_cases"].sum() + df_returnable["quantity_cases"].sum() - rsplit_cases_before) < 0.01, \
    "cases lost in the split - stop and check"
assert not (df["Category2"] == RSPLIT_CAT).any(), "Returnable rows still in df - stop and check"
assert set(df["Product_Code"]).isdisjoint(set(df_returnable["Product_Code"])), \
    "a product code is in both drinks and Returnable - stop and check"

# ---- Returnable weekly series, built the same way as Prep cells 21-22 build the drinks ----
#      (Monday-start weeks; zero-filled from each item's first sale to the last week)
rsplit_keys = ["Category2", "Sub_Category", "Brand", "Product_Code"]
rsplit_weekly = (df_returnable
                 .assign(ds=pd.to_datetime(df_returnable["visit_date"]).dt.to_period("W").dt.start_time)
                 .groupby(rsplit_keys + ["ds"], as_index=False)["quantity_cases"].sum()
                 .rename(columns={"quantity_cases": "y"}))
rsplit_pieces = []
for rsplit_key, rsplit_g in rsplit_weekly.groupby(rsplit_keys):          # one loop per packaging item
    rsplit_weeks = pd.date_range(rsplit_g["ds"].min(), rsplit_last_week, freq="W-MON")
    rsplit_s = rsplit_g.set_index("ds")["y"].reindex(rsplit_weeks, fill_value=0)   # absent weeks -> 0
    rsplit_pieces.append(pd.DataFrame({
        "unique_id": "/".join(str(v) for v in rsplit_key),              # same style as Y_product, e.g. Returnable/Returnable/Returnable/350592
        "ds": rsplit_weeks,
        "y": rsplit_s.values,
    }))
Y_returnable = pd.concat(rsplit_pieces, ignore_index=True)

assert abs(Y_returnable["y"].sum() - df_returnable["quantity_cases"].sum()) < 0.01, "Returnable cases lost - stop and check"
assert Y_returnable["unique_id"].nunique() == df_returnable["Product_Code"].nunique(), "Returnable item count wrong"
assert (Y_returnable.groupby("unique_id")["ds"].max() == rsplit_last_week).all(), "a Returnable series does not end on the last week"

Y_returnable.to_csv("../eabl parsed csv/Y_returnable.csv", index=False)

# ---- Report ----
print(f"Rows      before: {rsplit_rows_before:,}  ->  drinks (df): {len(df):,}  +  Returnable (df_returnable): {len(df_returnable):,}")
print(f"Cases     before: {rsplit_cases_before:,.2f}  ->  drinks: {df['quantity_cases'].sum():,.2f}  +  Returnable: {df_returnable['quantity_cases'].sum():,.2f}")
print(f"Products  before: {rsplit_prods_before}  ->  drinks: {df['Product_Code'].nunique()}  +  Returnable: {df_returnable['Product_Code'].nunique()}")
print(f"Categories left in df: {sorted(df['Category2'].unique())}")
print(f"\nSaved Y_returnable.csv: {Y_returnable.shape}  ({Y_returnable['unique_id'].nunique()} items, "
      f"{Y_returnable['ds'].min().date()} to {Y_returnable['ds'].max().date()})")
print("Drinks and Returnable together: ../eabl parsed csv/sales_products_forecast_ready.csv (unchanged)")


In [ ]:

# Null count and percentage for every column, sorted highest first
null_summary = pd.DataFrame({
    "nulls": df.isna().sum(),
    "pct": df.isna().mean() * 100
}).sort_values("nulls", ascending=False)

print(null_summary[null_summary["nulls"] > 0])

In [ ]:
for c in ["Division", "Area", "Territory", "Route", "global Channel", "Sub-Channel"]:
    print(c, df[c].nunique(), df[c].isna().sum())


In [ ]:
df.isna().sum()

In [ ]:
df.shape

## 1.4 — Cost of Over- vs. Under-Forecasting: Data-Driven Investigation

**Objective:** Determine whether real cost signals exist in the data to distinguish the cost
of overstocking vs. understocking, per product category/brand — since these costs are not
symmetric (a spoiled case of beer vs. a missed high-margin spirits sale are very different losses).

### Method
Three checks were run against the project data:
1. Return rate by `Category2`, using `quantity_cases` and `returns_cases` from
   `hierarchical_forecast_dataset.csv` — an attempt to measure overstock/spoilage directly.
2. Average price per case by `Category2`, from `product_master_cleaned.csv` — a proxy for the
   cost of a missed sale (understock cost).
3. Real "Expired [Brand]" write-off volume, extracted from `unmatched_products_review.csv`
   (`return_quantity` field) — since these transactions were excluded from the main pipeline.

### Findings

| Check | Result |
|---|---|
| Return rate by category | Beer, RTD, Spirits all show **0%** — `returns_cases` only tracks bottle/crate deposits ("Returnable" = 72.7%), not spoilage. This field cannot be used as an overstock signal for alcohol categories. |
| Price per case | Spirits (~20,502/case) is **~6.4x** the price of Beer (~3,210/case) — a real, usable proxy for understock cost (lost margin per missed sale). |
| Real write-offs | 163 total units expired across 9 SKUs: Guinness (76), White Cap (28), Tusker Ndimu (27), Baileys (22), Gordons Premix (10). |

### Conclusion

- No usable spoilage-cost field exists in the current pipeline — this has to be sourced from
  finance directly.
- Real spoilage is **not confined to Beer**. Baileys and Gordons (both classified under
  `Category2 = Spirits`) had confirmed write-offs — the blanket assumption "Spirits doesn't
  spoil" is too broad and specifically wrong for cream-based/premix products.
- The correct level to apply an overstock/understock cost adjustment is **Brand**, not
  `Category2` — at minimum, treat Guinness, White Cap, Tusker Ndimu, Baileys, and Gordons
  as a separate, higher-overstock-risk group from the rest of Spirits.
- Sample size for the write-off evidence is small (163 units / 16 months) — enough to
  redirect attention to specific brands, not enough to compute a statistically confident
  dollar cost ratio on its own.

### Next steps
- [ ] Bring this exact 5-brand list to finance/supply chain and request real holding,
      spoilage, and stockout-penalty cost figures for these brands specifically.
- [ ] Until real figures are available, use a placeholder lean: order conservatively for
      Beer + the 5 flagged brands; order more generously for the rest of Spirits/RTD.
- [ ] Proceed to Step 1.5 (success metrics + sign-off) in parallel — it does not depend on
      finance's response.


In [ ]:
# ---------- 1. Return rate by category (using returns_cases) ----------

by_cat = df.groupby("Category2").agg(
    total_sold=("quantity_cases", "sum"),
    total_returned=("returns_cases", "sum")
)
by_cat["return_rate_pct"] = by_cat["total_returned"] / by_cat["total_sold"] * 100
print("--- Return rate by category (overstock proxy attempt #1) ---")
print(by_cat.sort_values("return_rate_pct", ascending=False))
print()


In [ ]:
# ---------- 2. Price per case by category (understock proxy) ----------
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")
price_by_cat = pm.groupby("Category2")["FY26_EX_KBL_Price_Per_Case"].agg(["mean", "median", "count"])
print("--- Price per case by category (understock proxy) ---")
print(price_by_cat.sort_values("mean", ascending=False))
print()

# ---------- 3. Real expired/write-off volume by brand (overstock proxy #2) ----------
unmatched = pd.read_csv("../eabl parsed csv/unmatched_products_review.csv")
expired = unmatched[unmatched["product"].str.contains("Expired", case=False, na=False)]
print("--- Expired write-off volume by product (real overstock evidence) ---")
print(expired.groupby("product")["return_quantity"].sum().sort_values(ascending=False))



def adjusted_order_quantity(forecast, brand, category):
    risky_brands = ["Guinness", "White Cap", "Tusker Ndimu", "Baileys", "Gordons"]
    
    if category == "Beer" or brand in risky_brands:
        adjustment = 0.90
    else:
        adjustment = 1.10
    
    return forecast * adjustment

# Example
print(adjusted_order_quantity(100, "Tusker Lager", "Beer"))   # → 90.0 (Beer category, order less)
print(adjusted_order_quantity(100, "Baileys", "Spirits"))     # → 90.0 (flagged brand, order less)
print(adjusted_order_quantity(100, "Smirnoff", "Spirits"))    # → 110.0 (safe Spirits, order more)


In [ ]:
for c in ["Volume Grade", "Archi Type", "Cutsomer Type"]:
    print(f"--- {c} ---")
    print("nulls:", df[c].isna().sum(), f"({df[c].isna().mean()*100:.3f}%)")
    print("distinct values:", df[c].nunique())
    print(df[c].value_counts())
    print()



In [ ]:
print(df["Mode of Payment"].isna().sum())
print(df["Mode of Payment"].value_counts())

In [ ]:
# Does each Sub_Category belong to exactly one Category2?
nesting_check = pm.groupby("Sub_Category")["Category2"].nunique()
print("Sub_Categories spanning more than one Category2:")
print(nesting_check[nesting_check > 1])

# Does each New_Sub_Category2 belong to exactly one Sub_Category?
nesting_check2 = pm.groupby("New_Sub_Category2")["Sub_Category"].nunique()
print("New_Sub_Category2 values spanning more than one Sub_Category:")
print(nesting_check2[nesting_check2 > 1])

print()
print("Cardinality check (should increase going down the tree):")
for c in ["Category2", "Sub_Category", "New_Sub_Category2", "Brand"]:
    print(c, pm[c].nunique())


In [ ]:

df["visit_date"] = pd.to_datetime(df["visit_date"])
df["ds"] = df["visit_date"].dt.to_period("W").dt.start_time


def validate_S(Y_df, S_df, tags, bottom_spec, name):
    print(f"--- Validating {name} ---")
    print("S shape:", S_df.shape)

    # Make sure S_df is indexed by unique_id, not a default row number
    if "unique_id" in S_df.columns:
        S_df = S_df.set_index("unique_id")

    bottom_tag = "/".join(bottom_spec)
    bottom_ids = tags[bottom_tag]

    missing = [i for i in bottom_ids if i not in S_df.index]
    if missing:
        print(f"WARNING: {len(missing)} bottom ids not found in S_df index. Example missing id:", missing[0])
        print("S_df index sample:", list(S_df.index[:5]))
        print()
        return

    S_bottom = S_df.loc[bottom_ids]
    ok_rows = (S_bottom.sum(axis=1) == 1).all()
    ok_cols = (S_bottom.sum(axis=0) == 1).all()
    print("Bottom rows self-map correctly:", ok_rows)
    print("Bottom columns uniquely claimed:", ok_cols)

    if "unique_id" in Y_df.columns:
        # Pick a sample date; missing series that week = 0 sales, not an error
        sample_date = Y_df["ds"].iloc[0]
        y_t_raw = Y_df[Y_df["ds"] == sample_date].set_index("unique_id")["y"]

        b_t = y_t_raw.reindex(bottom_ids, fill_value=0)
        y_t = y_t_raw.reindex(S_df.index, fill_value=0)

        reconstructed = S_df.values @ b_t.values
        identity_holds = np.allclose(reconstructed, y_t.values)
        print("S @ b == y on sample date:", identity_holds)
    print()


# ---------- A. Product ----------
long_product = (
    df.groupby(["Category2", "Sub_Category", "Brand", "Product_Code", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_product = [
    ["Category2"],
    ["Category2", "Sub_Category"],
    ["Category2", "Sub_Category", "Brand"],
    ["Category2", "Sub_Category", "Brand", "Product_Code"],
]
Y_product, S_product, tags_product = aggregate(df=long_product, spec=spec_product)
validate_S(Y_product, S_product, tags_product, spec_product[-1], "Product")

# ---------- B. Geographic — DIAGNOSE FIRST ----------
route_check = df.groupby("Route")[["Division", "Area", "Territory"]].nunique()
problem_routes = route_check[(route_check["Territory"] > 1)]
print("Routes appearing under more than one Territory:")
print(problem_routes)
print()

# Composite key to fix any duplicate Route names across Territories
df["Route_full"] = df["Territory"] + " | " + df["Route"]

long_geo = (
    df.groupby(["Division", "Area", "Territory", "Route_full", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_geo = [
    ["Division"],
    ["Division", "Area"],
    ["Division", "Area", "Territory"],
    ["Division", "Area", "Territory", "Route_full"],
]
Y_geo, S_geo, tags_geo = aggregate(df=long_geo, spec=spec_geo)
validate_S(Y_geo, S_geo, tags_geo, spec_geo[-1], "Geographic (fixed)")

# ---------- C. Channel ----------
long_channel = (
    df.groupby(["global Channel", "Sub-Channel", "segment_grouped", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_channel = [
    ["global Channel"],
    ["global Channel", "Sub-Channel"],
    ["global Channel", "Sub-Channel", "segment_grouped"],
]
Y_channel, S_channel, tags_channel = aggregate(df=long_channel, spec=spec_channel)
validate_S(Y_channel, S_channel, tags_channel, spec_channel[-1], "Channel")

# ---------- D. Grouped — DIAGNOSE FIRST ----------
combo_check = df[["Category2", "segment_grouped"]].drop_duplicates()
print("Distinct Category2 x segment_grouped combinations found:")
print(combo_check.sort_values(["Category2", "segment_grouped"]))
print()

long_grouped = (
    df.groupby(["Category2", "segment_grouped", "ds"], as_index=False)["quantity_cases"]
    .sum().rename(columns={"quantity_cases": "y"})
)
spec_grouped = [
    ["Category2"],
    ["segment_grouped"],
    ["Category2", "segment_grouped"],
]
Y_grouped, S_grouped, tags_grouped = aggregate(df=long_grouped, spec=spec_grouped)
validate_S(Y_grouped, S_grouped, tags_grouped, spec_grouped[-1], "Grouped")



In [ ]:
# ==========================================================================
# STEP 8 - Fill missing weeks with 0 (Product hierarchy only)
# A product with no sales in a week currently has NO row for that week.
# Here each product gets one row for EVERY week from its FIRST sale to the
# last week of the data; the weeks that were absent get y = 0.
# Weeks before a product's first sale are NOT filled (it wasn't on sale yet).
# Only long_product changes. df and the Geo/Channel/Grouped tables are untouched.
# ==========================================================================
keys = ["Category2", "Sub_Category", "Brand", "Product_Code"]
last_week = long_product["ds"].max()                 # last complete week (22 Jun 2026 after step 3)

rows_before  = len(long_product)
total_before = long_product["y"].sum()
before_fill  = long_product.copy()                   # kept only for the example below

pieces = []
for key, g in long_product.groupby(keys):
    all_weeks = pd.date_range(g["ds"].min(), last_week, freq="W-MON")   # first sale -> last week, every Monday
    s = g.set_index("ds")["y"].reindex(all_weeks, fill_value=0)          # absent weeks become 0
    piece = s.rename_axis("ds").reset_index()
    for col, val in zip(keys, key):
        piece[col] = val
    pieces.append(piece)

long_product = pd.concat(pieces, ignore_index=True)[keys + ["ds", "y"]]

# ---- Checks ---------------------------------------------------------------
added = len(long_product) - rows_before
print(f"Product-weeks before: {rows_before:,}   after: {len(long_product):,}   zero weeks added: {added:,}")
print(f"Total cases before: {total_before:,.2f}   after: {long_product['y'].sum():,.2f}   "
      f"(must be equal: {np.isclose(total_before, long_product['y'].sum())})")
n_gaps = long_product.groupby(keys)["ds"].apply(lambda d: (d.diff().dropna() != pd.Timedelta(weeks=1)).sum()).sum()
print(f"Products with a week still missing: {n_gaps}  (must be 0)")
print(f"Every product now ends on: {long_product.groupby(keys)['ds'].max().unique()}")

# ---- Example: Ciroc (625018) before and after ------------------------------
pick = lambda t: t[t["Product_Code"].astype(str) == "625018"].set_index("ds")["y"]
example = pd.DataFrame({"Before (rows that existed)": pick(before_fill),
                        "After (zero-filled)": pick(long_product)})
example.index = example.index.strftime("%d %b %Y")
example["Before (rows that existed)"] = example["Before (rows that existed)"].map(
    lambda v: "(no row)" if pd.isna(v) else round(v, 2))
example["After (zero-filled)"] = example["After (zero-filled)"].round(2)
print("\nExample - Ciroc 625018, week by week:")
display(example)

# ---- Rebuild the Product hierarchy from the zero-filled weeks --------------
Y_product, S_product, tags_product = aggregate(df=long_product, spec=spec_product)
validate_S(Y_product, S_product, tags_product, spec_product[-1], "Product")
print("Y_product rows:", len(Y_product), "| series:", Y_product["unique_id"].nunique())


In [ ]:
# ==========================================================================
# STEP 8 CHECK - Did the zero-fill do exactly what we agreed, for EVERY product?
# Read-only: uses before_fill, long_product, keys, last_week, Y_product from
# the Step 8 cell. Nothing is changed or saved.
# ==========================================================================
before = before_fill.copy()
after  = long_product.copy()

# ---- 1. One line per product: before vs after ------------------------------
b = before.groupby(keys).agg(first_sale=("ds", "min"), weeks_with_sales=("ds", "size"), total_before=("y", "sum"))
a = after.groupby(keys).agg(starts=("ds", "min"), ends=("ds", "max"), weeks_after=("ds", "size"),
                            zero_weeks=("y", lambda v: (v == 0).sum()), total_after=("y", "sum"))
per_product = b.join(a, how="outer").reset_index()
per_product["zeros_added"]        = per_product["weeks_after"] - per_product["weeks_with_sales"]
per_product["% weeks with 0"]     = (per_product["zero_weeks"] / per_product["weeks_after"] * 100).round(1)
per_product["starts at 1st sale"] = per_product["starts"] == per_product["first_sale"]
per_product["ends 22 Jun"]        = per_product["ends"] == last_week
per_product["total unchanged"]    = np.isclose(per_product["total_before"], per_product["total_after"])

# ---- 2. The rules we agreed, checked for all products ----------------------
merged = after.merge(before, on=keys + ["ds"], how="left", suffixes=("", "_before"))
weeks_gap = after.groupby(keys)["ds"].apply(lambda d: (d.diff().dropna() != pd.Timedelta(weeks=1)).sum())
y_weeks_gap = Y_product.groupby("unique_id")["ds"].apply(
    lambda d: (pd.to_datetime(d).sort_values().diff().dropna() != pd.Timedelta(weeks=1)).sum())

rules = pd.DataFrame({
    "Check": [
        "Same products before and after (no product added or lost)",
        "Every product starts at its own first sale (no zeros before launch)",
        "Every product ends on the last full week (22 Jun 2026)",
        "No week missing inside any product",
        "Every week that HAD sales still has exactly the same number",
        "Every added week is exactly 0",
        "No product-week appears twice",
        "No negative values",
        "Total cases unchanged for every product",
        "Every Y_product series (all 319 levels) has no missing week",
    ],
    "Result": [
        len(b) == len(a) == per_product[keys].drop_duplicates().shape[0],
        per_product["starts at 1st sale"].all(),
        per_product["ends 22 Jun"].all(),
        (weeks_gap == 0).all(),
        np.allclose(merged.loc[merged["y_before"].notna(), "y"], merged.loc[merged["y_before"].notna(), "y_before"]),
        (merged.loc[merged["y_before"].isna(), "y"] == 0).all(),
        not after.duplicated(keys + ["ds"]).any(),
        (after["y"] >= 0).all(),
        per_product["total unchanged"].all(),
        (y_weeks_gap == 0).all(),
    ],
})
rules["Result"] = rules["Result"].map({True: "PASS", False: "FAIL"})
print(f"Products: {len(per_product)}   |   zero weeks added in total: {per_product['zeros_added'].sum():,}")
display(rules)

# ---- 3. Who got zeros, and how many ----------------------------------------
n_filled = (per_product["zeros_added"] > 0).sum()
print(f"\n{n_filled} of {len(per_product)} products got at least one 0-week; "
      f"{len(per_product) - n_filled} sold every week and are unchanged.")

show = per_product.sort_values("zeros_added", ascending=False)[
    ["Category2", "Brand", "Product_Code", "first_sale", "weeks_with_sales", "weeks_after",
     "zeros_added", "% weeks with 0", "total_before", "total_after"]].copy()
show["first_sale"] = show["first_sale"].dt.strftime("%d %b %Y")
show[["total_before", "total_after"]] = show[["total_before", "total_after"]].round(2)
show.columns = ["Category", "Brand", "Product code", "First sale (week of)", "Weeks with sales",
                "Weeks after fill", "Zeros added", "% weeks that are 0", "Total cases before", "Total cases after"]
show = show.reset_index(drop=True); show.index += 1
with pd.option_context("display.max_rows", None, "display.width", 250):
    display(show)

# ---- 4. Look at any single product week by week ----------------------------
def inspect_product(code):
    """Week-by-week before vs after for one Product_Code, e.g. inspect_product('625018')."""
    pick = lambda t: t[t["Product_Code"].astype(str) == str(code)].set_index("ds")["y"]
    t = pd.DataFrame({"Before (rows that existed)": pick(before), "After (zero-filled)": pick(after)})
    if t.empty:
        print(f"No product with code {code}"); return
    name = per_product.loc[per_product["Product_Code"].astype(str) == str(code), "Brand"].iloc[0]
    t["What happened"] = np.where(t["Before (rows that existed)"].isna(), "0 added (no sale that week)", "kept as it was")
    t["Before (rows that existed)"] = t["Before (rows that existed)"].map(lambda v: "(no row)" if pd.isna(v) else round(v, 2))
    t["After (zero-filled)"] = t["After (zero-filled)"].round(2)
    t.index = t.index.strftime("%d %b %Y")
    print(f"\n{name} - {code}: {(t['What happened'] != 'kept as it was').sum()} zero weeks added out of {len(t)} weeks")
    with pd.option_context("display.max_rows", None):
        display(t)

inspect_product("625018")      # Ciroc - change the code to look at any other product


In [ ]:
S_product.to_csv("../eabl parsed csv/S_product.csv")
Y_product.to_csv("../eabl parsed csv/Y_product.csv", index=False)

S_geo.to_csv("../eabl parsed csv/S_geo.csv")
Y_geo.to_csv("../eabl parsed csv/Y_geo.csv", index=False)

S_channel.to_csv("../eabl parsed csv/S_channel.csv")
Y_channel.to_csv("../eabl parsed csv/Y_channel.csv", index=False)

S_grouped.to_csv("../eabl parsed csv/S_grouped.csv")
Y_grouped.to_csv("../eabl parsed csv/Y_grouped.csv", index=False)


# STEP 0 – Rules of the honest test (read before running anything below)

**Source:** Hyndman & Athanasopoulos, *Forecasting: Principles and Practice*, 2nd ed. (OTexts, 2018), §3.4 and §4.2.
These rules apply to every product in the Product hierarchy (Category2 → Sub_Category → Brand → Product_Code, 188 drinks).

| Week | Monday date | Role |
|---|---|---|
| 1 – 53 | 7 Apr 2025 – **6 Apr 2026** | Training + cross-validation (models may learn from these) |
| 54 – 66 | 13 Apr 2026 – 6 Jul 2026 | **TEST – sealed** (13 weeks = forecast horizon h) |
| 67 – 79 | 13 Jul 2026 – 5 Oct 2026 | Final forecast (refit on weeks 1 – 66 only after the test is scored) |

*Dates come from the `ds` column of `../eabl parsed csv/Y_product.csv` (66 Monday-labelled weeks).*

**(a) The seal is Monday 6 Apr 2026 (week 53).** Nothing used to build, choose or tune a model may use sales after this week.
> FPP2 §3.4: "The accuracy of forecasts can only be determined by considering how well a model performs on new data that were not used when fitting the model."

**(b) Weeks 54 – 66 are never edited or deleted.** Cleaning steps (F6 – F9) may *flag* these weeks but never change `y`, fill zeros, or remove rows. Any cleaning only touches weeks 1 – 53, keeps the original values, and saves to a new file.

**(c) Settings are locked before the test is opened:**
- `F2_H = 13`, `F2_MIN_SALES = 3` (F2 cell, this Prep notebook)
- Cross-validation: `h = 13`, `n_windows = 2`, `step_size = 13` (F1 notebook)

These are not changed after seeing test results.
> FPP2 §4.2: "Formalising and documenting the decision rules and assumptions … can promote consistency, as the same rules can be implemented repeatedly."

**(d) The test is scored ONCE,** at the end. No changing models or settings and re-scoring.

**(e) Judgement forecasts** (products with no sales before 6 Apr 2026) use only information known by 6 Apr 2026. Planners do not see weeks 54 – 66 first.
> FPP2 §4.2: implement "a systematic approach" and "Segregate forecasters and users".

**(f) Existing F1 accuracy numbers are CV scores, NOT test scores.** They were computed with cross-validation on all 66 weeks, so the last CV round used weeks 54 – 66, and the same errors chose each product's model. This applies to the WAPEs in F1 (standard group CV, Tier 1 / Tier 2 overall WAPE) and to `tier1_weekly_vs_cumulative.csv` and `final_forecast_all_skus.csv`.

**(g) Earlier cells that looked at ALL 66 weeks** (flags / diagnostics only – no sales values were changed). Each must be re-checked on weeks 1 – 53 before it is used to clean data or choose a model:
- Cleaning notebook: cell 31 (Category2 taken from the current product master), cell 34 (large-order limit = 5 × each SKU's 99th percentile over all dates), cell 35 ("normal for this customer" compares with any other date, including later ones), cells 40 – 41 (Sunday-spike rule; all 8 rows of `known_spikes.csv` are in the week of 8 Dec 2025, inside training)
- Prep notebook: cells 47 – 48 (structural breaks), cells 73 – 80 (STL seasonal strength) *(cell numbers updated 2026-11-04)*


In [ ]:
# ==========================================================================
# STEP D2 - Count trading days per week (a full week = Mon-Sat = 6 days)
#
# WHY (FPP2 3.2 calendar adjustments; 5.4 "Trading days"; 10.1 hierarchy):
#   If some weeks had fewer trading days (holidays, missing reports), their sales would
#   look low only because the week was shorter. FPP2 then models sales PER trading day
#   (3.2) or adds trading days as a predictor (5.4). This cell measures whether that is
#   needed. It also lists near-empty days, public holidays and Sunday sales per week,
#   for the holiday checks (F6, F7) and the spike/drop screen (F8).
#
# WHERE EACH INPUT COMES FROM:
#   df        -> Prep cell 5 (sales + customer master join), cell 9 (unmatched customers
#                dropped), cell 21 (visit_date made a date, ds = Monday of each week).
#                One row per sales line. Columns used:
#                  visit_date     - sale date (from sales_products_forecast_ready.csv, read in cell 1)
#                  quantity_cases - quantity in cases (from the Cleaning notebook)
#                  customer_code  - the outlet
#                  DMS Customer Code - customer master key (cell 5), used only to confirm cell 9 ran
#   Y_product -> Prep cell 22 (zero-filled, re-aggregated). Used only to check the weeks match.
#   pd        -> pandas, imported in Prep cell 0.
#   holidays  -> the 'holidays' package, imported in Prep cell 0 (Kenya public holidays).
#   display   -> Jupyter's built-in table display.
#
# WHAT IT CHANGES: nothing in df, Y_product or S_product (read-only on them).
# WHAT IT WRITES:  ../eabl parsed csv/trading_days_per_week.csv  (NEW file, one row per week)
# All new names start with d2_ / D2_ (no other Prep cell uses them).
# ==========================================================================
D2_NEAR_EMPTY = 0.5      # a Mon-Sat day with < 50% of the usual sales lines for that weekday

# ---- Safety checks: stop (and save nothing) if the inputs are not what we expect ----
assert {"visit_date", "ds", "quantity_cases", "customer_code", "DMS Customer Code"}.issubset(df.columns), \
    "`df` is not Prep's joined daily table - re-run Prep cells 1 to 21 first"
assert df["DMS Customer Code"].notna().all(), "unmatched customers still in df - run Prep cell 9 first"
assert pd.api.types.is_datetime64_any_dtype(df["visit_date"]), "visit_date is not a date - run Prep cell 21 first"
assert {"unique_id", "ds", "y"}.issubset(Y_product.columns), "`Y_product` missing - run Prep cells 21 and 22 first"

# ---- 1. One row per calendar day: sales lines, cases, outlets ----
d2_day = (df.groupby(df["visit_date"].dt.normalize())
            .agg(rows=("quantity_cases", "size"),
                 cases=("quantity_cases", "sum"),
                 customers=("customer_code", "nunique")))
d2_day["dow"]  = d2_day.index.dayofweek                          # 0 = Mon ... 5 = Sat, 6 = Sun
d2_day["week"] = d2_day.index.to_period("W").start_time          # Monday of the week (same rule as cell 21)
d2_usual = d2_day[d2_day["dow"] < 6].groupby("dow")["rows"].median()   # usual lines per weekday
d2_day["share_of_usual"] = (d2_day["rows"] / d2_day["dow"].map(d2_usual)).round(2)
d2_day["near_empty"] = (d2_day["dow"] < 6) & (d2_day["share_of_usual"] < D2_NEAR_EMPTY)

# ---- 2. Kenya public holidays for the years in the data ----
d2_hol = holidays.Kenya(years=range(d2_day.index.year.min(), d2_day.index.year.max() + 1))

# ---- 3. One row per week (Mon-Sun) ----
d2_rows = []
for d2_wk in pd.date_range(d2_day["week"].min(), d2_day["week"].max(), freq="W-MON"):
    d2_monsat = pd.date_range(d2_wk, periods=6, freq="D")         # the 6 trading days of this week
    d2_g      = d2_day[d2_day["week"] == d2_wk]
    d2_sold   = set(d2_g.index[d2_g["dow"] < 6])                  # Mon-Sat days that had sales
    d2_rows.append({
        "week_start":       d2_wk,
        "trading_days":     len(d2_sold),
        "missing_days":     ", ".join(d2_d.strftime("%a %d %b") for d2_d in d2_monsat if d2_d not in d2_sold),
        "near_empty_days":  int(d2_g["near_empty"].sum()),
        "near_empty_list":  ", ".join(d2_d.strftime("%a %d %b") for d2_d in d2_g.index[d2_g["near_empty"]]),
        "holidays_mon_sat": sum(d2_d in d2_hol for d2_d in d2_monsat),
        "holiday_names":    "; ".join(f"{d2_d.strftime('%a %d %b')} {d2_hol.get(d2_d)}" for d2_d in d2_monsat if d2_d in d2_hol),
        "sunday_sales":     bool((d2_g["dow"] == 6).any()),
        "cases":            d2_g["cases"].sum(),
    })
d2_weeks = pd.DataFrame(d2_rows)
d2_weeks["cases_vs_median_week"] = (d2_weeks["cases"] / d2_weeks["cases"].median()).round(2)

# ---- Checks: same weeks as Y_product, no cases lost ----
d2_y_weeks = pd.DatetimeIndex(pd.to_datetime(Y_product["ds"]).unique()).sort_values()
assert pd.DatetimeIndex(d2_weeks["week_start"]).equals(d2_y_weeks), \
    "weeks here differ from Y_product's weeks - re-run Prep cells 21 to 24"
assert abs(d2_weeks["cases"].sum() - df["quantity_cases"].sum()) < 0.01, "cases lost while counting - stop and check"
d2_weeks["cases"] = d2_weeks["cases"].round(2)

# ---- Results ----
print(f"Weeks: {len(d2_weeks)}  ({d2_weeks['week_start'].min().date()} to {d2_weeks['week_start'].max().date()})  - same weeks as Y_product")
print("\nTrading days per week (Mon-Sat days with at least one sale) -> number of weeks:")
print(d2_weeks["trading_days"].value_counts().sort_index().to_string())
d2_short = d2_weeks[d2_weeks["trading_days"] < 6]
print("\nWeeks with FEWER than 6 trading days:", "none" if d2_short.empty else "")
if not d2_short.empty:
    display(d2_short[["week_start", "trading_days", "missing_days", "holiday_names", "cases_vs_median_week"]])

print("\nUsual sales lines per weekday (median):",
      {pd.Timestamp(2025, 4, 7 + k).strftime("%a"): int(v) for k, v in d2_usual.items()})
d2_ne = d2_day[d2_day["near_empty"]].copy()
d2_ne["day"]     = d2_ne.index.strftime("%a %d %b %Y")
d2_ne["holiday"] = [d2_hol.get(d2_d, "") for d2_d in d2_ne.index]
d2_ne = d2_ne.merge(d2_weeks[["week_start", "cases_vs_median_week"]], left_on="week", right_on="week_start", how="left")
print(f"\nNear-empty trading days (< {D2_NEAR_EMPTY:.0%} of the usual lines for that weekday): "
      f"{len(d2_ne)} days in {d2_ne['week'].nunique()} weeks")
display(d2_ne[["day", "rows", "customers", "cases", "share_of_usual", "holiday", "cases_vs_median_week"]]
        .rename(columns={"cases_vs_median_week": "whole week vs median"}).round(2))

print(f"\nPublic holidays on Mon-Sat: {int(d2_weeks['holidays_mon_sat'].sum())} days in "
      f"{int((d2_weeks['holidays_mon_sat'] > 0).sum())} weeks   |   weeks with Sunday sales: {int(d2_weeks['sunday_sales'].sum())}")

# ---- Save the weekly calendar (NEW file; used later by F6, F7 and F8) ----
d2_weeks.to_csv("../eabl parsed csv/trading_days_per_week.csv", index=False)
print("\nSaved trading_days_per_week.csv:", d2_weeks.shape)

In [ ]:
# ==========================================================================
# STEP F2 - Label each product: active / late launch / stopped / unclear
# FPP2 4.6: new products need different handling and the reasoning must be documented.
# FPP2 12.7: very short series cannot be tested/cross-validated normally.
# FPP2 10.3: bottom-up sums product forecasts, so a stopped product's forecast
#            would add "phantom" cases to its brand, sub-category and category.
# READ-ONLY: uses Y_product, tags_product and df; changes none of them, saves nothing.
# Saving the labels is F4.
# ==========================================================================

F2_H         = 13   # 13 weeks = forecast horizon h and the locked test window (13 Apr - 6 Jul 2026)
F2_MIN_SALES = 3    # fewer selling weeks than this = too few to judge (judgement, see FPP2 12.7)

# ---- Safety checks: stop if the inputs are not the ones this cell expects ----
f2_bottom_key = "Category2/Sub_Category/Brand/Product_Code"   # product-level key made by aggregate() in Prep cell 22
assert f2_bottom_key in tags_product, "tags_product has no product level - re-run Prep cells 21 and 22 first"
assert {"unique_id", "ds", "y"}.issubset(Y_product.columns), "`Y_product` missing - re-run Prep cells 21 and 22 first"
assert {"Product_Code", "SKU", "Status"}.issubset(df.columns), "`df` is not Prep's joined table - re-run Prep cells 1 to 21"

# ---- Product-level rows only (203 products), taken from the zero-filled Y_product (Prep cell 22) ----
f2_bottom = Y_product[Y_product["unique_id"].isin(tags_product[f2_bottom_key])].copy()
f2_bottom["ds"] = pd.to_datetime(f2_bottom["ds"])
f2_weeks = pd.DatetimeIndex(sorted(Y_product["ds"].unique()))   # the 66 Monday week-starts, 7 Apr 2025 - 6 Jul 2026
f2_T = len(f2_weeks)                                             # 66
assert f2_bottom["unique_id"].nunique() == len(tags_product[f2_bottom_key]), "some products have no weekly rows"
assert (f2_bottom["y"] >= 0).all(), "negative weekly sales found - stop and check"

# ---- EABL's product master name (SKU) and status (Active/Inactive), carried in df from the Cleaning merge ----
f2_master = (df.assign(Product_Code=df["Product_Code"].astype(str))   # df's codes are numbers; unique_id ends in text
               .groupby("Product_Code")[["SKU", "Status"]].first())  # one SKU and one Status per code (checked)

f2_rows = []
for f2_uid, f2_g in f2_bottom.groupby("unique_id"):      # one loop per product, e.g. "Spirits/Rum/Captain Morgan Muckpit/799700"
    # weekly sales on the full 66-week calendar (weeks before the first sale = 0, only for counting here)
    f2_y    = f2_g.sort_values("ds").set_index("ds")["y"].reindex(f2_weeks, fill_value=0)
    f2_sold = np.flatnonzero(f2_y.values > 0)            # positions (0-65) of weeks with a sale
    f2_first, f2_last = f2_sold[0], f2_sold[-1]
    f2_lead = int(f2_first)                              # empty weeks BEFORE the first sale
    f2_tail = int(f2_T - 1 - f2_last)                    # empty weeks AFTER the last sale
    f2_run = f2_gap = 0                                  # longest run of 0-weeks BETWEEN first and last sale
    for f2_v in f2_y.values[f2_first:f2_last + 1]:
        f2_run = f2_run + 1 if f2_v <= 0 else 0
        f2_gap = max(f2_gap, f2_run)
    f2_n = len(f2_sold)                                  # number of weeks with a sale

    # ---- The rules, checked in this order (first match wins) ----
    if f2_lead >= F2_H and f2_tail >= F2_H:
        f2_label, f2_why = "unclear", "started late AND no sale in the last 13+ weeks"
    elif f2_n < F2_MIN_SALES and f2_lead < f2_T - F2_H:
        f2_label, f2_why = "unclear", f"sold in only {f2_n} week(s) - too few to judge"
    elif f2_tail >= F2_H and f2_tail > f2_gap:
        f2_label, f2_why = "stopped", f"no sale for the last {f2_tail} weeks (longest earlier pause {f2_gap})"
    elif f2_lead >= F2_H and f2_lead > f2_gap:
        f2_label, f2_why = "late launch", f"first sale in week {f2_lead + 1} of {f2_T} (longest later pause {f2_gap})"
    elif f2_lead >= F2_H or f2_tail >= F2_H:
        f2_label, f2_why = "unclear", f"quiet for 13+ weeks at the start/end, but it has paused {f2_gap} weeks before"
    else:
        f2_label, f2_why = "active", "sells from the start to the end of the data"

    f2_code = f2_uid.split("/")[-1]                      # last part of unique_id = Product_Code
    f2_rows.append({
        "unique_id": f2_uid, "Product_Code": f2_code,
        "SKU": f2_master["SKU"].get(f2_code, ""), "master_status": f2_master["Status"].get(f2_code, ""),
        "first_sale_week": f2_weeks[f2_first].date(), "last_sale_week": f2_weeks[f2_last].date(),
        "weeks_before_first_sale": f2_lead, "weeks_after_last_sale": f2_tail,
        "longest_pause_weeks": f2_gap, "weeks_with_sales": f2_n, "weeks_on_sale": f2_T - f2_lead,
        "total_cases": round(float(f2_y.sum()), 2), "label": f2_label, "why": f2_why,
    })
f2_labels = pd.DataFrame(f2_rows)

# ---- Checks: one row per product, and no cases lost ----
assert len(f2_labels) == len(tags_product[f2_bottom_key]) and f2_labels["Product_Code"].is_unique
assert abs(f2_labels["total_cases"].sum() - f2_bottom["y"].sum()) < 1, "cases lost while labelling - stop and check"

# ---- Report ----
print(f"Products labelled: {len(f2_labels)}   weeks in data: {f2_T} "
      f"({f2_weeks[0].date()} to {f2_weeks[-1].date()})")
print("\nLabel counts:"); print(f2_labels["label"].value_counts().to_string())
print("\nShare of all cases sold, by label (%):")
print((f2_labels.groupby("label")["total_cases"].sum() / f2_labels["total_cases"].sum() * 100).round(2).to_string())
print("\nLabel vs EABL product-master Status:")
print(pd.crosstab(f2_labels["label"], f2_labels["master_status"]).to_string())
f2_show = ["Product_Code", "SKU", "master_status", "first_sale_week", "last_sale_week",
           "weeks_with_sales", "longest_pause_weeks", "total_cases", "why"]
for f2_lab in ["late launch", "stopped", "unclear"]:
    print(f"\n--- {f2_lab} ---")
    display(f2_labels.loc[f2_labels["label"] == f2_lab, f2_show].sort_values("first_sale_week"))


In [ ]:
# ==========================================================================
# STEP F2 - FIX 1: BACKTEST labels = the SAME F2 rules, using ONLY weeks 1-53
#                  (up to the seal, Monday 6 Apr 2026 - STEP 0 rule (a))
#
# WHY (FPP2 3.4): "The accuracy of forecasts can only be determined by considering how
#   well a model performs on new data that were not used when fitting the model."
#   The F2 cell above sees all 66 weeks, so some products get a label that could only
#   be known AFTER 6 Apr 2026 (e.g. Kenya Cane Citrus is "stopped" only because its 13
#   test weeks are empty). For the backtest, labels must be what was knowable on 6 Apr 2026.
#
# TWO LABEL SETS FROM NOW ON (STEP 0 rule (a)):
#   f2_labels          -> F2 cell above (66 weeks) -> FINAL labels, only for the final forecast (weeks 67-79)
#   f2_labels_backtest -> THIS cell     (53 weeks) -> BACKTEST labels, for CV and the sealed test (weeks 54-66)
#
# WHERE EACH INPUT COMES FROM (all already in memory - nothing is read from disk):
#   Y_product     -> Prep cell 23 (STEP 8 zero-fill, then aggregate()): long table unique_id / ds / y,
#                    66 Monday weeks, every level of the Product hierarchy (drinks only, after R1 in cell 10)
#   tags_product  -> Prep cell 23 (same aggregate() call): dict level-name -> list of unique_ids
#   f2_bottom_key -> F2 cell above: "Category2/Sub_Category/Brand/Product_Code" (the product level)
#   f2_master     -> F2 cell above: SKU name per Product_Code (from df). Only SKU is used here -
#                    master Status is NOT copied (July 2026 master = after the seal, leak row 7)
#   f2_labels     -> F2 cell above: the 66-week (FINAL) labels, used only for the proof and the comparison
#   F2_H, F2_MIN_SALES -> F2 cell above: 13 and 3 (locked, STEP 0 rule (c))
#   pd, np        -> Prep cell 0 (imports);  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing. Y_product, tags_product, df and f2_labels are only read.
# WHAT IT WRITES:  no file (saving both label sets is F4). New table in memory: f2_labels_backtest.
# All new names start with f2b_ / F2B_ so nothing clashes with the F2 cell (f2_ / F2_).
# ==========================================================================

F2B_CUTOFF = pd.Timestamp("2026-04-06")   # the seal: Monday of week 53 (STEP 0 rule (a))

# ---- Safety checks: stop if the inputs are not the ones this cell expects ----
assert "f2_labels" in globals() and "f2_master" in globals(), "run the F2 cell (STEP F2 - Label each product) first"
assert (F2_H, F2_MIN_SALES) == (13, 3), "F2 settings changed - they are locked (STEP 0 rule (c))"
assert f2_bottom_key in tags_product, "tags_product has no product level - re-run Prep cells 22 and 23"
assert {"unique_id", "ds", "y"}.issubset(Y_product.columns), "`Y_product` missing - re-run Prep cells 22 and 23"
assert len(f2_labels) == len(tags_product[f2_bottom_key]), "f2_labels is stale - re-run the F2 cell"

# ---- The F2 rules, written ONCE as a function so the 53- and 66-week runs are guaranteed identical ----
def f2b_label_products(f2b_Y, f2b_weeks):
    """f2b_Y: product-level rows (unique_id, ds, y); f2b_weeks: the Monday weeks this run may see.
    Same counting and the same 6 rules, in the same order, as the F2 cell above."""
    f2b_T = len(f2b_weeks)
    f2b_rows = []
    for f2b_uid, f2b_g in f2b_Y.groupby("unique_id"):
        f2b_y = f2b_g.sort_values("ds").set_index("ds")["y"].reindex(f2b_weeks, fill_value=0)
        f2b_sold = np.flatnonzero(f2b_y.values > 0)           # positions of weeks with a sale
        f2b_first, f2b_last = f2b_sold[0], f2b_sold[-1]
        f2b_lead = int(f2b_first)                              # empty weeks BEFORE the first sale
        f2b_tail = int(f2b_T - 1 - f2b_last)                   # empty weeks AFTER the last sale
        f2b_run = f2b_gap = 0                                  # longest run of 0-weeks BETWEEN first and last sale
        for f2b_v in f2b_y.values[f2b_first:f2b_last + 1]:
            f2b_run = f2b_run + 1 if f2b_v <= 0 else 0
            f2b_gap = max(f2b_gap, f2b_run)
        f2b_n = len(f2b_sold)                                  # number of weeks with a sale

        # ---- The rules, in the F2 cell's order (first match wins) ----
        if f2b_lead >= F2_H and f2b_tail >= F2_H:
            f2b_label, f2b_why = "unclear", "started late AND no sale in the last 13+ weeks"
        elif f2b_n < F2_MIN_SALES and f2b_lead < f2b_T - F2_H:
            f2b_label, f2b_why = "unclear", f"sold in only {f2b_n} week(s) - too few to judge"
        elif f2b_tail >= F2_H and f2b_tail > f2b_gap:
            f2b_label, f2b_why = "stopped", f"no sale for the last {f2b_tail} weeks (longest earlier pause {f2b_gap})"
        elif f2b_lead >= F2_H and f2b_lead > f2b_gap:
            f2b_label, f2b_why = "late launch", f"first sale in week {f2b_lead + 1} of {f2b_T} (longest later pause {f2b_gap})"
        elif f2b_lead >= F2_H or f2b_tail >= F2_H:
            f2b_label, f2b_why = "unclear", f"quiet for 13+ weeks at the start/end, but it has paused {f2b_gap} weeks before"
        else:
            f2b_label, f2b_why = "active", "sells from the start to the end of the data"

        f2b_code = f2b_uid.split("/")[-1]                      # last part of unique_id = Product_Code
        f2b_rows.append({
            "unique_id": f2b_uid, "Product_Code": f2b_code, "SKU": f2_master["SKU"].get(f2b_code, ""),
            "first_sale_week": f2b_weeks[f2b_first].date(), "last_sale_week": f2b_weeks[f2b_last].date(),
            "weeks_before_first_sale": f2b_lead, "weeks_after_last_sale": f2b_tail,
            "longest_pause_weeks": f2b_gap, "weeks_with_sales": f2b_n, "weeks_on_sale": f2b_T - f2b_lead,
            "total_cases": round(float(f2b_y.sum()), 2), "label": f2b_label, "why": f2b_why,
        })
    return pd.DataFrame(f2b_rows)

# ---- Product-level rows from Y_product (Prep cell 23), and the 66 weeks ----
f2b_all = Y_product[Y_product["unique_id"].isin(tags_product[f2_bottom_key])].copy()
f2b_all["ds"] = pd.to_datetime(f2b_all["ds"])
f2b_weeks_all = pd.DatetimeIndex(sorted(f2b_all["ds"].unique()))
assert len(f2b_weeks_all) == 66 and F2B_CUTOFF in f2b_weeks_all, "Y_product does not have the expected 66 weeks"

# ---- PROOF the function = the F2 cell: on all 66 weeks it must give exactly f2_labels ----
f2b_proof = f2b_label_products(f2b_all, f2b_weeks_all).set_index("unique_id")
f2b_ref   = f2_labels.set_index("unique_id")
assert f2b_proof.index.sort_values().equals(f2b_ref.index.sort_values()), "product list differs from f2_labels"
f2b_cols = ["label", "weeks_before_first_sale", "weeks_after_last_sale", "longest_pause_weeks", "weeks_with_sales"]
assert (f2b_proof.loc[f2b_ref.index, f2b_cols] == f2b_ref[f2b_cols]).all().all(), \
    "the rules here do not match the F2 cell - stop, do not use this cell"

# ---- BACKTEST run: weeks 1-53 only (up to and including 6 Apr 2026) ----
f2b_weeks = f2b_weeks_all[f2b_weeks_all <= F2B_CUTOFF]                        # 53 weeks
f2b_cut   = f2b_all[f2b_all["ds"] <= F2B_CUTOFF]
f2b_known = f2b_cut.groupby("unique_id")["y"].sum().loc[lambda s: s > 0].index   # products with a sale by 6 Apr 2026
f2_labels_backtest = f2b_label_products(f2b_cut[f2b_cut["unique_id"].isin(f2b_known)], f2b_weeks)
f2_labels_backtest["cutoff"]    = F2B_CUTOFF.date()                          # stamp: which weeks these labels saw
f2_labels_backtest["label_set"] = "BACKTEST"

# ---- Checks ----
assert len(f2b_weeks) == 53
assert f2_labels_backtest["unique_id"].is_unique
assert abs(f2_labels_backtest["total_cases"].sum() - f2b_cut["y"].sum()) < 1, "cases lost while labelling - stop and check"
assert pd.to_datetime(f2_labels_backtest["last_sale_week"]).max() <= F2B_CUTOFF, "a test week was used - stop"
f2b_unknown = sorted(set(f2b_ref.index) - set(f2_labels_backtest["unique_id"]))   # first sale after 6 Apr 2026

# ---- Report ----
print(f"BACKTEST labels: {len(f2_labels_backtest)} products, {len(f2b_weeks)} weeks "
      f"({f2b_weeks[0].date()} to {f2b_weeks[-1].date()})")
print("Proof: the same rules on all 66 weeks reproduce f2_labels exactly - OK")
print("\nLabel counts (BACKTEST, 53 weeks):"); print(f2_labels_backtest["label"].value_counts().to_string())

print(f"\nNot known on 6 Apr 2026 (first sale after the seal): {len(f2b_unknown)} products")
display(f2b_ref.loc[f2b_unknown, ["SKU", "first_sale_week", "label"]]
        .rename(columns={"label": "FINAL label (66 wks)"}))

f2b_cmp = (f2_labels_backtest.set_index("unique_id")[["SKU", "label"]]
           .rename(columns={"label": "BACKTEST (53 wks)"})
           .join(f2b_ref["label"].rename("FINAL (66 wks)")))
f2b_changed = f2b_cmp[f2b_cmp["BACKTEST (53 wks)"] != f2b_cmp["FINAL (66 wks)"]]
print("\nBACKTEST vs FINAL label (rows = 53 weeks, columns = 66 weeks):")
print(pd.crosstab(f2b_cmp["BACKTEST (53 wks)"], f2b_cmp["FINAL (66 wks)"]).to_string())
print(f"\nProducts whose label changes once the test weeks are seen: {len(f2b_changed)}")
display(f2b_changed.sort_values(["BACKTEST (53 wks)", "FINAL (66 wks)"]))


In [ ]:
# ==========================================================================
# STEP F3 - How much history does each product have BEFORE the seal (6 Apr 2026)?
#           full / short / too new  -> tells F1 which method can be used per product
#
# WHY:
#   FPP2 3.4 (time series cross-validation): "Since it is not possible to obtain a reliable
#     forecast based on a small training set, the earliest observations are not considered
#     as test sets."  -> a product needs enough weeks BEFORE each CV round to be tested.
#   FPP2 12.7: "with short series, there is not enough data to allow some observations to be
#     withheld for testing purposes, and even time series cross validation can be difficult
#     to apply. The AICc is particularly useful here"  -> short products: choose by AICc.
#   FPP2 4.6: "Judgmental forecasting is usually the only available method for new product
#     forecasting, as historical data are unavailable."  -> too new products: judgement.
#
# THE GROUPS (CV settings locked in STEP 0 rule (c): h = 13, n_windows = 2, step_size = 13):
#   CV round 1 learns weeks 1-27 and predicts 28-40; round 2 learns 1-40 and predicts 41-53.
#   full    = at least 13 selling-period weeks by the end of round 1's training (week 27)
#             = 39+ weeks before the seal -> both CV rounds usable -> CV picks the model
#   short   = 1-38 weeks before the seal -> CV not reliable -> AICc picks the model (FPP2 12.7)
#   too new = 0 weeks (first sale after 6 Apr 2026) -> no model, judgement (FPP2 4.6),
#             scored separately at Step 8
#   F3_MIN_FIT = 13 is OUR choice (one quarter = the forecast horizon). FPP2 12.7 warns that
#   fixed minimum sample sizes are rules of thumb, so 13 is documented here, not claimed as a law.
#
# LEAK CHECK: only the FIRST SALE DATE is used, capped at the seal. A first sale on or before
#   6 Apr 2026 was known on that day; a first sale after it gives 0 either way. Nothing from
#   weeks 54-66 changes a group. (Closes F2 leak row 5 with Fix 1; prepares F1 fix O at Step 7.)
#
# WHERE EACH INPUT COMES FROM (all already in memory - nothing is read from disk):
#   f2_labels          -> F2 cell (Prep cell 28): all 188 products, column first_sale_week
#   f2_labels_backtest -> Fix 1 cell (Prep cell 29): 181 products known by 6 Apr 2026,
#                         columns label and weeks_on_sale (= weeks from first sale to the seal)
#   F2B_CUTOFF         -> Fix 1 cell (Prep cell 29): Timestamp 2026-04-06, the seal
#   f2_T               -> F2 cell (Prep cell 28): 66, the number of weeks in Y_product
#   Y_product          -> Prep cell 23: used only to list the 66 Monday weeks
#   pd, np             -> Prep cell 0 (imports);  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing. f2_labels, f2_labels_backtest and Y_product are only read.
# WHAT IT WRITES:  no file (saving is F4). New table in memory: f3_history (188 rows).
# All new names start with f3_ / F3_ so nothing clashes with other cells.
# ==========================================================================

F3_CV_H       = 13   # CV horizon h            (locked, STEP 0 rule (c))
F3_CV_WINDOWS = 2    # CV n_windows            (locked, STEP 0 rule (c))
F3_MIN_FIT    = 13   # weeks a model needs to learn from before a CV round (our documented choice)

# ---- Safety checks: stop if the inputs are not the ones this cell expects ----
assert "f2_labels" in globals(), "run the F2 cell (Prep cell 28) first"
assert "f2_labels_backtest" in globals() and "F2B_CUTOFF" in globals(), "run the Fix 1 cell (Prep cell 29) first"
assert {"unique_id", "Product_Code", "SKU", "first_sale_week"}.issubset(f2_labels.columns), "f2_labels is not the F2 table"
assert {"unique_id", "label", "weeks_on_sale"}.issubset(f2_labels_backtest.columns), "f2_labels_backtest is not the Fix 1 table"
assert (f2_labels_backtest["label_set"] == "BACKTEST").all(), "f2_labels_backtest is not the BACKTEST table"

# ---- The weeks: 66 in Y_product, 53 up to the seal, and where each CV round's training ends ----
f3_weeks_all = pd.DatetimeIndex(sorted(pd.to_datetime(Y_product["ds"]).unique()))   # 66 Mondays
assert len(f3_weeks_all) == f2_T == 66, "Y_product does not have the expected 66 weeks"
f3_train_weeks = f3_weeks_all[f3_weeks_all <= F2B_CUTOFF]                            # 53 weeks
f3_T_train = len(f3_train_weeks)                                                     # 53
f3_cv1_end = f3_T_train - F3_CV_WINDOWS * F3_CV_H                                    # 27 = last week round 1 learns from
f3_cv2_end = f3_T_train - F3_CV_H                                                    # 40 = last week round 2 learns from
assert (f3_T_train, f3_cv1_end, f3_cv2_end) == (53, 27, 40)

# ---- One row per product (all 188), from the F2 table's first sale date ----
f3_history = f2_labels[["unique_id", "Product_Code", "SKU", "first_sale_week"]].copy()
f3_first = pd.to_datetime(f3_history["first_sale_week"])
f3_lead = np.searchsorted(f3_weeks_all.values, f3_first.values)       # weeks before the first sale (0 = sold in week 1)

# weeks of history available at each point (0 if the product had not started yet)
f3_history["weeks_in_train"]   = np.clip(f3_T_train - f3_lead, 0, None)   # by the seal (week 53)
f3_history["weeks_by_cv1_end"] = np.clip(f3_cv1_end - f3_lead, 0, None)   # by week 27 (round 1)
f3_history["weeks_by_cv2_end"] = np.clip(f3_cv2_end - f3_lead, 0, None)   # by week 40 (round 2)
f3_history["cv_rounds_usable"] = ((f3_history["weeks_by_cv1_end"] >= F3_MIN_FIT).astype(int)
                                  + (f3_history["weeks_by_cv2_end"] >= F3_MIN_FIT).astype(int))

# ---- The groups ----
f3_history["history"] = np.select(
    [f3_history["weeks_in_train"] == 0,
     f3_history["weeks_by_cv1_end"] < F3_MIN_FIT],
    ["too new", "short"], default="full")
f3_history["history_plan"] = f3_history["history"].map({
    "full":    "both CV rounds usable -> CV picks the model",
    "short":   "too little history for CV -> AICc picks the model (FPP2 12.7)",
    "too new": "no sales before 6 Apr 2026 -> judgement forecast, scored separately (FPP2 4.6)",
})

# backtest label from Fix 1 (the 7 products not known on 6 Apr 2026 have none)
f3_history = f3_history.merge(f2_labels_backtest[["unique_id", "label"]]
                              .rename(columns={"label": "label_backtest"}), on="unique_id", how="left")
f3_history["label_backtest"] = f3_history["label_backtest"].fillna("not known yet")
f3_history["cutoff"] = F2B_CUTOFF.date()

# ---- Checks ----
assert len(f3_history) == len(f2_labels) and f3_history["unique_id"].is_unique
assert f3_history["history"].notna().all() and f3_history["history"].isin(["full", "short", "too new"]).all()
f3_chk = f3_history.merge(f2_labels_backtest[["unique_id", "weeks_on_sale"]], on="unique_id", how="inner")
assert len(f3_chk) == len(f2_labels_backtest), "products missing between F3 and Fix 1"
assert (f3_chk["weeks_in_train"] == f3_chk["weeks_on_sale"]).all(), \
    "weeks_in_train does not match Fix 1's weeks_on_sale - stop and check"
assert set(f3_history.loc[f3_history["history"] == "too new", "unique_id"]) == \
       set(f2_labels["unique_id"]) - set(f2_labels_backtest["unique_id"]), "too new != products unknown on 6 Apr 2026"
assert (f3_history.loc[f3_history["history"] == "full", "cv_rounds_usable"] == 2).all()

# ---- Report ----
print(f"F3 history groups: {len(f3_history)} products | seal {F2B_CUTOFF.date()} | "
      f"{f3_T_train} training weeks | CV round 1 learns weeks 1-{f3_cv1_end}, round 2 learns 1-{f3_cv2_end}")
print("Check: weeks_in_train matches Fix 1's weeks_on_sale for all", len(f3_chk), "known products - OK")
print(f"\nGroup counts (full = {F3_MIN_FIT + F3_CV_WINDOWS * F3_CV_H}+ weeks before the seal):")
print(f3_history["history"].value_counts().reindex(["full", "short", "too new"]).to_string())

f3_cases = f2_labels_backtest.set_index("unique_id")["total_cases"]          # cases sold in weeks 1-53 only
f3_share = f3_history.set_index("unique_id").join(f3_cases).fillna({"total_cases": 0}) \
                     .groupby("history")["total_cases"].sum()
print("\nShare of cases sold in weeks 1-53, by group (%):")
print((f3_share / f3_share.sum() * 100).reindex(["full", "short", "too new"]).fillna(0).round(2).to_string())

print("\nHistory group x BACKTEST label:")
print(pd.crosstab(f3_history["history"], f3_history["label_backtest"]).to_string())

f3_show = ["Product_Code", "SKU", "first_sale_week", "weeks_in_train", "weeks_by_cv1_end",
           "weeks_by_cv2_end", "cv_rounds_usable", "label_backtest"]
for f3_grp in ["short", "too new"]:
    print(f"\n--- {f3_grp}: {f3_history.loc[f3_history['history'] == f3_grp, 'history_plan'].iloc[0]} ---")
    display(f3_history.loc[f3_history["history"] == f3_grp, f3_show].sort_values("first_sale_week"))


In [ ]:
Y_product

In [ ]:
Y_product

In [ ]:
S_product

In [ ]:
df

## Naive Baseline Forecast (WAPE)

### What this cell does

1. **Load and aggregate to weekly.** Reads the cleaned dataset and converts each date to the week it falls in.
2. **Group by brand and week.** Sums sales so there is exactly **one row per brand per week**.
3. **Build the naive forecast.** `.shift(1)` takes last week's actual sales and uses them as this week's forecast. This is the *naive method*: no model and no math, just the assumption that this week will look like last week.
4. **Measure the error.** Compares the naive forecast against actual sales and rolls the total error into one percentage, the **Weighted Absolute Percentage Error (WAPE)**:

$$
\text{WAPE} = \frac{\sum |\text{Actual} - \text{Forecast}|}{\sum |\text{Actual}|}
$$

   That one number shows how far off the simplest possible approach would have been across the full history.

### Why it matters

This WAPE is the **baseline every real forecasting model has to beat.**

- If "repeat last week" is already ~25% off, a real model needs to land meaningfully **below 25%** to be worth using.
- If it can't, the extra work of hierarchical forecasting doesn't pay off.
- Without a baseline, "our model is good" has no reference point. *Good compared to what?*

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
df["week"] = df["visit_date"].dt.to_period("W")

# Aggregate sales to Brand x Week
weekly = df.groupby(["Brand", "week"])["quantity_cases"].sum().reset_index()

# The naive forecast for this week = whatever actually happened last week
weekly["naive_forecast"] = weekly.groupby("Brand")["quantity_cases"].shift(1)

# Error = how far off that naive guess was from the real number
weekly["abs_error"] = (weekly["quantity_cases"] - weekly["naive_forecast"]).abs()

# WAPE = total error as a percentage of total actual sales
wape = weekly["abs_error"].sum() / weekly["quantity_cases"].sum() * 100
print("Naive baseline WAPE:", wape)


In [ ]:
kenya_holidays = holidays.Kenya(years=[2025, 2026])
for date, name in sorted(kenya_holidays.items()):
    print(date, name)

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])
df["is_month_end_payday"] = df["visit_date"].dt.day >= (df["visit_date"].dt.days_in_month - 2)

In [ ]:
holiday_df = pd.DataFrame(list(kenya_holidays.items()), columns=["visit_date", "holiday_name"])
holiday_df["visit_date"] = pd.to_datetime(holiday_df["visit_date"])

df = df.merge(holiday_df, on="visit_date", how="left")
df["is_holiday"] = df["holiday_name"].notna()


In [ ]:
df.isna().sum()

In [ ]:
df["holiday_name"] = df["holiday_name"].fillna("normalday")

In [ ]:
df.isna().sum()

In [ ]:
df["holiday_name"].unique()

In [ ]:
df["day_of_week"] = df["visit_date"].dt.dayofweek        # 0=Monday, 6=Sunday
df["is_weekend"] = df["day_of_week"].isin([5, 6])
df["week_of_year"] = df["visit_date"].dt.isocalendar().week
df["month"] = df["visit_date"].dt.month
df["quarter"] = df["visit_date"].dt.quarter
df["is_month_start"] = df["visit_date"].dt.is_month_start
df["is_month_end"] = df["visit_date"].dt.is_month_end

# Cyclical encoding, so the model knows December and January are adjacent
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Days to/from nearest holiday
holiday_dates = pd.to_datetime(sorted(df.loc[df["is_holiday"], "visit_date"].unique()))
def days_to_nearest_holiday(date):
    diffs = (holiday_dates - date).days
    return diffs[np.abs(diffs).argmin()]
df["days_to_nearest_holiday"] = df["visit_date"].apply(days_to_nearest_holiday)


In [ ]:
df

In [ ]:
months = list(range(1, 13))
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

sin_vals = [np.sin(2*np.pi*m/12) for m in months]
cos_vals = [np.cos(2*np.pi*m/12) for m in months]

fig, ax = plt.subplots(figsize=(7,7))

# draw the unit circle for reference
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.sin(theta), np.cos(theta), color='lightgray', linestyle='--', linewidth=1)

# connect months in calendar order, including Dec back to Jan, to show the loop
loop_sin = sin_vals + [sin_vals[0]]
loop_cos = cos_vals + [cos_vals[0]]
ax.plot(loop_sin, loop_cos, color='steelblue', linewidth=1.5, zorder=1)

ax.scatter(sin_vals, cos_vals, color='crimson', s=80, zorder=2)

for m, name, x, y in zip(months, month_names, sin_vals, cos_vals):
    ax.annotate(name, (x, y), textcoords='offset points', xytext=(8,8), fontsize=11, fontweight='bold')

ax.set_xlabel('month_sin')
ax.set_ylabel('month_cos')
ax.set_title('Months placed on a circle using sin/cos\n(Dec connects straight back to Jan)')
ax.set_aspect('equal')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig('month_cycle.png', dpi=150)
plt.show()


In [ ]:
import ruptures as rpt
import matplotlib.pyplot as plt

# ---------- Step 1: Diagnose what the real top-level label looks like ----------
print(Y_product.columns.tolist())
print(Y_product["unique_id"].unique()[:10])
print(list(tags_product.keys()))

# ---------- Step 2: Pick the actual top-level series safely ----------
candidate_labels = ["Total"] + list(Y_product["unique_id"].unique()[:5])

top_level_id = None
for label in candidate_labels:
    subset = Y_product[Y_product["unique_id"] == label]
    if len(subset) > 5:
        top_level_id = label
        break

if top_level_id is None:
    raise ValueError("Could not find a usable top-level series — check the printed unique_id list above.")

print("Using top-level series:", top_level_id)

total_series = Y_product[Y_product["unique_id"] == top_level_id].sort_values("ds")
values = total_series["y"].values
print("Number of data points:", len(values))

# ---------- Step 3: Only run ruptures if there's actually enough data ----------
if len(values) < 10:
    print("Not enough data points to run breakpoint detection reliably.")
else:
    algo = rpt.Pelt(model="rbf").fit(values)
    breakpoints = algo.predict(pen=10)

    print("Detected breakpoints (index positions):", breakpoints)
    print(
        "Corresponding dates:",
        total_series["ds"].iloc[[b - 1 for b in breakpoints[:-1]]].tolist(),
    )

    rpt.display(values, breakpoints)
    plt.title(f"Detected structural breaks — {top_level_id} weekly sales")
    plt.show()

    # Repeat the same check for the other three hierarchies
    for Y_df, name in [(Y_geo, "Geographic"), (Y_channel, "Channel"), (Y_grouped, "Grouped")]:
        candidates = ["Total"] + list(Y_df["unique_id"].unique()[:5])
        for label in candidates:
            s = Y_df[Y_df["unique_id"] == label].sort_values("ds")
            if len(s) > 5:
                vals = s["y"].values
                algo = rpt.Pelt(model="rbf").fit(vals)
                bkps = algo.predict(pen=10)
                print(f"{name} ({label}) breakpoints:", bkps)
                break


In [ ]:
total_by_week = Y_product[Y_product["unique_id"].isin(["Beer", "RTD", "Spirits", "Returnable"])]
total_by_week = total_by_week.groupby("ds")["y"].sum().reset_index().sort_values("ds")

values = total_by_week["y"].values
algo = rpt.Pelt(model="rbf").fit(values)
breakpoints = algo.predict(pen=10)
print("True company-wide breakpoints:", breakpoints)


In [ ]:

# ==================================================================
# PART 1 — What CAN be done: correctly separate "true missing"
# (a data gap) from "confirmed zero" (a real, trustworthy zero sale),
# for EACH product individually, only within its own active window.
# ==================================================================

def classify_series_gaps(df, id_col, date_col, value_col, full_calendar):
    """
    For every bottom-level series (e.g. each Product_Code), compare its
    own active date range against the full calendar and classify each
    period as: true_missing (no row at all) or confirmed_zero/nonzero
    (a real recorded value).
    """
    results = []
    for uid, group in df.groupby(id_col):
        node_start, node_end = group[date_col].min(), group[date_col].max()
        expected = full_calendar[(full_calendar >= node_start) & (full_calendar <= node_end)]
        observed = group.set_index(date_col)[value_col].reindex(expected)

        results.append({
            "unique_id": uid,
            "active_from": node_start,
            "active_to": node_end,
            "true_missing_periods": observed.isna().sum(),
            "confirmed_zero_periods": (observed == 0).sum(),
            "nonzero_periods": (observed > 0).sum(),
        })
    return pd.DataFrame(results)


# Build the weekly date column (Mondays) and a matching calendar (also Mondays)
df["visit_date"] = pd.to_datetime(df["visit_date"])
df["ds"] = df["visit_date"].dt.to_period("W").dt.start_time

# FIX: "W-MON" so the calendar anchors on Mondays, matching df["ds"] exactly
full_calendar = pd.date_range(df["ds"].min(), df["ds"].max(), freq="W-MON")

bottom_df = df.groupby(["Product_Code", "ds"], as_index=False)["quantity_cases"].sum()

gap_classification = classify_series_gaps(
    bottom_df, id_col="Product_Code", date_col="ds",
    value_col="quantity_cases", full_calendar=full_calendar
)

print("Products with the most real data gaps (worth investigating):")
print(gap_classification.sort_values("true_missing_periods", ascending=False).head(10))
print()
print("Overall summary:")
print(gap_classification[["true_missing_periods", "confirmed_zero_periods", "nonzero_periods"]].sum())


In [ ]:


# ==================================================================
# PART 2 — OPTIONAL heuristic: flag WEEKS THAT LOOK LIKE a possible
# stockout (a temporary drop to zero, followed by a recovery). This is
# a guess, not proof — real confirmation needs actual stock data,
# which does not exist in this project.
# ==================================================================

def flag_suspected_stockouts(series, min_run=2, recovery_ratio=0.7):
    """
    series: weekly quantities for ONE product, indexed by date,
    with true gaps already left as NaN (not filled with 0).
    Returns True for weeks that look like a plausible stockout dip.
    """
    vals = series.fillna(0)
    is_zero = (vals == 0)
    flags = pd.Series(False, index=series.index)

    i = 0
    while i < len(vals):
        if is_zero.iloc[i]:
            j = i
            while j < len(vals) and is_zero.iloc[j]:
                j += 1
            before = vals.iloc[max(0, i - 4):i].mean() if i > 0 else np.nan
            after = vals.iloc[j:j + 4].mean() if j < len(vals) else np.nan
            if (j - i) >= min_run and pd.notna(before) and before > 0 and pd.notna(after) and after >= recovery_ratio * before:
                flags.iloc[i:j] = True   # plausible stockout — NOT confirmed
            i = j
        else:
            i += 1
    return flags


# Example: run this on one specific product's weekly series
example_product = bottom_df["Product_Code"].iloc[0]
one_series = (
    bottom_df[bottom_df["Product_Code"] == example_product]
    .set_index("ds")["quantity_cases"]
    .reindex(full_calendar)   # now correctly aligned, Monday-to-Monday
)
suspected = flag_suspected_stockouts(one_series)
print(f"\nSuspected stockout weeks for {example_product}:")
print(suspected[suspected].index.tolist())


In [ ]:


# ==================================================================
# PART 3 — The documentation step: the one thing that IS fully
# actionable today, since real stockout detection needs data that
# does not exist in this project.
# ==================================================================

DATA_SCOPE_NOTE = (
    "NOTE: This forecasting pipeline predicts RECORDED SALES, not true "
    "market demand. No inventory/stockout feed exists in this project as "
    "of {date}. Zero-sale periods are NOT distinguished from stockout-"
    "censored periods. If stock/inventory data becomes available, revisit "
    "this step to implement proper censored-demand estimation before "
    "trusting forecasts for chronically under-supplied SKUs/Routes."
).format(date=pd.Timestamp.today().date())

print("\n" + DATA_SCOPE_NOTE)


In [ ]:
df.isna().sum()

## 4.1 — Missing Data Taxonomy (Structural Zero vs. Stockout vs. True Missing)

**What this section does:**
For every product, compares its actual sales history against a full expected
weekly calendar (within its own active lifetime) to classify each week as
either a real recorded sale, or a gap with no row at all. It also runs a
heuristic that flags weeks which *look like* a possible stockout (a dip in
sales followed by a recovery), and documents the limitations of both checks.

### Key findings
- Across the whole catalog (each product's own active lifetime, Prep cell 49 "Overall summary",
  updated 2026-11-04): **1,755 weeks** have no recorded row at all,
  **8,636 weeks** have a real recorded sale, and **0 weeks** show a
  confirmed `0` — because this dataset's parsing process only ever keeps
  rows with a positive quantity, so a genuine "sold nothing" week can't be
  distinguished from "no visit happened" or "out of stock."
- The products with the most gaps are almost entirely low-volume,
  niche/reserve products (e.g. premium whiskies) that naturally sell only
  a handful of times a year — this is expected behavior, not a data error.
- The "suspected stockout" heuristic mostly re-flags these same sparse
  products rather than finding new evidence — for products with very few
  sales, nearly any gap looks like a "dip and recovery" by chance. This is
  a known weakness of the heuristic at low sales volume, not proof of a
  real stockout.

### Significance
This project has **no inventory/stockout data**, so a real zero can never
be fully separated from a hidden stockout with the data currently
available. This section makes that limitation explicit and quantified,
rather than leaving it as a silent assumption. The practical takeaway is
not "go find more stockouts" — it's that the flagged sparse products
should be routed to intermittent-demand-appropriate forecasting methods
(Step 4.4 / 5.6), since standard models will handle them poorly regardless
of the stockout question.

**Scope statement:** this pipeline forecasts *recorded sales*, not true
market demand. Revisit this step if a real stock/inventory feed ever
becomes available.


In [ ]:
# ==================================================================
# VIEW 1 — Histogram: how many "missing" weeks do products typically have?
# ==================================================================
plt.figure(figsize=(8, 5))
plt.hist(gap_classification["true_missing_periods"], bins=30, color="steelblue", edgecolor="white")
plt.xlabel("True missing periods (weeks with no row at all)")
plt.ylabel("Number of products")
plt.title("How many products have how many data gaps?")
plt.tight_layout()
plt.show()


# ==================================================================
# BUILD stockout_summary — moved here, BEFORE anything that uses it
# ==================================================================
all_flags = []
for pid, group in bottom_df.groupby("Product_Code"):
    series = group.set_index("ds")["quantity_cases"].reindex(full_calendar)
    flags = flag_suspected_stockouts(series)
    flagged_weeks = flags[flags].index.tolist()
    if flagged_weeks:
        all_flags.append({
            "Product_Code": pid,
            "suspected_stockout_weeks": flagged_weeks,
            "count": len(flagged_weeks),
        })

stockout_summary = pd.DataFrame(all_flags).sort_values("count", ascending=False)
print(stockout_summary.shape)
print(stockout_summary.head(10))


# ==================================================================
# VIEW 2 — Bar chart: the 15 products with the most suspected stockouts
# ==================================================================
top15 = stockout_summary.head(15)

plt.figure(figsize=(9, 6))
plt.barh(top15["Product_Code"].astype(str), top15["count"], color="crimson")
plt.xlabel("Number of suspected stockout weeks")
plt.ylabel("Product Code")
plt.title("Top 15 products by suspected stockout weeks (heuristic, not confirmed)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


# ==================================================================
# VIEW 3 — corrected: robust date lookups instead of fragile .get()
# ==================================================================
def plot_product_timeline(product_code, ax):
    series = (
        bottom_df[bottom_df["Product_Code"] == product_code]
        .groupby("ds")["quantity_cases"].sum()   # ensure one value per date, no duplicates
        .reindex(full_calendar)                   # guarantees index is EXACTLY full_calendar
    )
    flags = flag_suspected_stockouts(series).reindex(full_calendar, fill_value=False)

    is_sale = series.fillna(0) > 0
    colors = np.where(
        is_sale, "seagreen",
        np.where(flags, "crimson", "lightgray")
    )

    ax.bar(full_calendar, [1] * len(full_calendar), color=colors, width=6)
    ax.set_yticks([])
    ax.set_title(f"Product {product_code}", loc="left", fontsize=10)


example_products = stockout_summary.head(4)["Product_Code"].tolist()

fig, axes = plt.subplots(len(example_products), 1, figsize=(12, 2 * len(example_products)), sharex=True)
for ax, pid in zip(axes, example_products):
    plot_product_timeline(pid, ax)

plt.xlabel("Week")
fig.suptitle("Week-by-week timeline: green = sold, gray = no row, red = suspected stockout", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
stockout_summary

In [ ]:
print(bottom_df["Product_Code"].unique()[:10])
print(bottom_df.shape)
print(stockout_summary.shape)

# Demand Classification: Technical Summary

…consistent), **intermittent** (rare, but consistent size when it happens), **erratic** (frequent, but unpredictable size), or **lumpy** (rare *and* unpredictable — the hardest case to forecast).

## Step 2 — Apply it to every individual product

This loops through every `Product_Code`, builds its full weekly history (gaps included as real gaps, not zeros), runs the classification function on it, and collects the results into one table. It then counts how many products fall into each bucket.

This is the part where we expect to find real "intermittent"/"lumpy" products, based on what we already discovered (the Reserve whiskies).

> **Updated by F5 (Prep cell 61):** each product's history now **starts at its first sale**, so
> weeks before a product existed are not counted as "no sale" (FPP2 §12.2: Croston's interval is
> "the time between" two sales; Syntetos, Boylan & Croston 2005). The 1.32 / 0.49 cut-offs are unchanged.

## Step 3 — The "don't just assume" check

This repeats the exact same classification on the bottom level of the other three hierarchies:

- Individual **Routes**
- Individual **Channel–Segment** combinations
- Individual **Category × Segment** cells

It uses the `S` / `Y` / `tags` objects you already built back in Step 2.3.

This tests the expectation that aggregation smooths things out, rather than taking it on faith:

- If these come back overwhelmingly **smooth**, that confirms it.
- If any come back **intermittent** or **lumpy**, that's a genuine exception worth knowing about before you build forecasts for it.

> **Action:** Run this and share all four `value_counts` outputs. That tells us exactly which series need Croston/SBA-style treatment versus standard methods going into F1.

---

# Plain-Language Version

## Step 1 — Sorting products into 4 simple groups

Think of every product as answering two questions:

1. **"How often do you sell?"** Every week, or only once in a while?
2. **"When you do sell, is it always about the same amount, or totally random each time?"**

Combining the answers to those two questions puts every product into one of four groups:

| Group | How often it sells | How much it sells | Difficulty |
|---|---|---|---|
| **Smooth** | Often | Similar amount each time | Easy to predict |
| **Intermittent** | Rarely | Similar amount each time | Moderate |
| **Erratic** | Often | Jumps around unpredictably | Moderate |
| **Lumpy** | Rarely | Unpredictable | Hardest to predict |

## Step 2 — Checking every single product

This runs that same check on every product one at a time and counts how many landed in each of the four groups.

We expect to find several products in the "rare" groups (intermittent/lumpy), like the Reserve whiskies we spotted earlier, which only sell a few times a year.

## Step 3 — Double-checking instead of assuming

Earlier we said that adding lots of products together usually smooths things out. In other words, a whole Route or a whole Channel probably sells steadily, even if some individual products inside it are rare sellers.

Step 3 doesn't just assume that's true. It runs the same 4-group check on **Routes**, **Channels**, and **Category + Segment** combinations too, to make sure nothing rare is hiding at those bigger levels as well.

## Why we do all this

Once we know which group each product, Route, or Channel falls into, we know which forecasting method to use for it. A rare seller needs a different approach than a steady one.

Sorting this out now means the next step (actually building forecasts) starts with the right tool for each situation instead of guessing.

In [ ]:

# ==================================================================
# STEP 1 — Define the classification rule (Syntetos & Boylan, 2005)
# ==================================================================
def classify_demand(series):
    """
    series: weekly quantities for ONE product/node, with true gaps as NaN.
    Returns one of: smooth, intermittent, erratic, lumpy, no_demand.
    """
    nonzero = series.dropna()
    nonzero = nonzero[nonzero > 0]
    if len(nonzero) == 0:
        return "no_demand"

    adi = len(series) / len(nonzero)                    # avg gap between sales
    cv2 = (nonzero.std() / nonzero.mean()) ** 2          # variability of sale size

    if adi < 1.32 and cv2 < 0.49:
        return "smooth"
    elif adi >= 1.32 and cv2 < 0.49:
        return "intermittent"
    elif adi < 1.32 and cv2 >= 0.49:
        return "erratic"
    else:
        return "lumpy"


In [ ]:
# ==========================================================================
# STEP F5: DEMAND CLASSIFICATION STARTS AT EACH PRODUCT'S LAUNCH
#          (the weeks BEFORE a product's first sale are no longer counted)
#
# THE PROBLEM (cell above, classify_demand):
#   ADI = len(series) / number of selling weeks. The series is reindexed to the WHOLE calendar,
#   so weeks before a product existed count as "weeks without a sale".
#   Example: a product launched in week 40 that sold in all 27 weeks since has
#   ADI = 66 / 27 = 2.44 -> "intermittent" / "lumpy" -> Croston toolbox,
#   but from launch it is ADI = 27 / 27 = 1.00 -> "smooth" / "erratic" -> standard toolbox.
#
# WHY (credible sources):
#   FPP2 12.2 (p.466), Croston's method: a_i is "the time between q_(i-1) and q_i", i.e. the gap
#     BETWEEN two sales. Weeks before the first sale are not a gap between sales.
#   Syntetos, Boylan & Croston (2005), J. Oper. Res. Soc. 56(5), 495-503 (the ADI 1.32 / CV2 0.49
#     rule in the cell above): ADI is the average interval between demands - same reasoning.
#   The ADI 1.32 / CV2 0.49 cut-offs and CV2 are NOT changed.
#
# HOW: this cell keeps the original rule as classify_demand_calendar and defines classify_demand
#   again so that it first cuts each series to start at its first week with a sale (> 0), then
#   calls the ORIGINAL rule unchanged. Weeks AFTER the last sale are kept (a stopped product's
#   zeros are real; stopped products are handled by the F2 labels).
#   Cells 61, 62 (Fix 3), 63 (3b) and 64 (F4) call classify_demand, so after this cell they use
#   the launch-based rule when re-run, for ALL four sets (cv1, cv2, backtest, final).
#
# LEAK RULES: the launch week is the first sale INSIDE the weeks each set may see (each set's
#   series is already cut at its own cut-off by cells 61-63), so no later week is used.
#
# WHERE EACH INPUT COMES FROM (already in memory):
#   classify_demand -> Prep cell 60 (the original Syntetos-Boylan-Croston rule)
#   bottom_df, full_calendar -> Prep cell 49: weekly cases per Product_Code; the 66 Monday weeks
#   f2b_weeks_all, F2B_CUTOFF -> Fix 1 (Prep cell 29): the 66 weeks; the seal 2026-04-06
#   f3_cv1_end, f3_cv2_end    -> F3 (Prep cell 30): 27 and 40 = last week of CV round 1 / round 2
#   pd, np -> Prep cell 0 (imports);  display -> Jupyter built-in
#
# WHAT IT CHANGES: only which function the name classify_demand points to (in memory).
#   The original stays available as classify_demand_calendar. No table is changed.
# WHAT IT WRITES:  nothing. (Re-running cell 61 afterwards re-saves classification_product.csv,
#   and F4 re-saves product_routing_labels.csv - both are their own files, as before.)
# All new names start with f5_ / F5_ (plus classify_demand_calendar).
# ==========================================================================

# ---- Keep the ORIGINAL rule once (safe to run this cell twice) ----
if "classify_demand_calendar" not in globals():
    classify_demand_calendar = classify_demand          # the cell-60 function, unchanged
assert classify_demand_calendar.__doc__ and "true gaps as NaN" in classify_demand_calendar.__doc__, \
    "classify_demand_calendar is not the cell-60 rule - re-run cell 60, then this cell"

def classify_demand(series):
    """Launch-based version: cut the series to start at its first week with a sale (> 0),
    then apply the ORIGINAL cell-60 rule (classify_demand_calendar) unchanged."""
    f5_sold = series[series > 0]
    if len(f5_sold) == 0:
        return classify_demand_calendar(series)                 # no sale at all -> "no_demand", as before
    return classify_demand_calendar(series.loc[f5_sold.index.min():])

# ---- Safety checks on the inputs used for the report ----
assert {"Product_Code", "ds", "quantity_cases"}.issubset(bottom_df.columns) and len(full_calendar) == 66, \
    "bottom_df / full_calendar missing - run Prep cell 49"
assert len(f2b_weeks_all) == 66 and (f3_cv1_end, f3_cv2_end) == (27, 40), "run Fix 1 (cell 29) and F3 (cell 30) first"

# ---- PROOF: for a product that sold in week 1, both rules must give the same class ----
f5_series = {pid: g.set_index("ds")["quantity_cases"].reindex(full_calendar) for pid, g in bottom_df.groupby("Product_Code")}
f5_week1 = [pid for pid, s in f5_series.items() if s.iloc[0] > 0]
assert all(classify_demand(f5_series[p]) == classify_demand_calendar(f5_series[p]) for p in f5_week1), \
    "the launch rule changes products that sold from week 1 - stop and check"

# ---- Report: old (whole calendar) vs new (from launch), for each set, each set's own weeks only ----
F5_SETS = {"cv1": f2b_weeks_all[f3_cv1_end - 1], "cv2": f2b_weeks_all[f3_cv2_end - 1],
           "backtest": F2B_CUTOFF, "final": f2b_weeks_all[-1]}
F5_FAMILY = {"smooth": "standard", "erratic": "standard", "intermittent": "intermittent", "lumpy": "intermittent"}
f5_rows = []
for f5_set, f5_cut in F5_SETS.items():
    for f5_pid, f5_s in f5_series.items():
        f5_s_cut = f5_s[f5_s.index <= f5_cut]
        if not (f5_s_cut > 0).any():
            continue                                             # not launched by this cut-off
        f5_launch = f5_s_cut[f5_s_cut > 0].index.min()
        f5_rows.append({"set": f5_set, "Product_Code": str(f5_pid), "launch_week": f5_launch.date(),
                        "weeks_before_launch": int((f5_s_cut.index < f5_launch).sum()),
                        "class_old": classify_demand_calendar(f5_s_cut), "class_new": classify_demand(f5_s_cut)})
f5_cmp = pd.DataFrame(f5_rows)
f5_cmp["family_old"] = f5_cmp["class_old"].map(F5_FAMILY)
f5_cmp["family_new"] = f5_cmp["class_new"].map(F5_FAMILY)
assert f5_cmp["class_new"].isin(F5_FAMILY.keys()).all(), "a launched product got no class - stop and check"

print("F5 - classify_demand now starts at each product's launch (original kept as classify_demand_calendar)")
print(f"Proof: {len(f5_week1)} products that sold from week 1 keep exactly the same class - OK\n")
f5_sum = (f5_cmp.assign(class_changed=f5_cmp["class_old"] != f5_cmp["class_new"],
                        family_changed=f5_cmp["family_old"] != f5_cmp["family_new"])
          .groupby("set", sort=False).agg(products=("Product_Code", "size"),
                                          launched_late=("weeks_before_launch", lambda s: int((s > 0).sum())),
                                          class_changed=("class_changed", "sum"), family_changed=("family_changed", "sum")))
print("Per set (each set uses only its own weeks):")
display(f5_sum)
print("\nFamily counts, old -> new:")
display(pd.DataFrame({f"{s} old": f5_cmp[f5_cmp["set"] == s]["family_old"].value_counts() for s in F5_SETS}
                     | {f"{s} new": f5_cmp[f5_cmp["set"] == s]["family_new"].value_counts() for s in F5_SETS})
        .fillna(0).astype(int)[[f"{s} {v}" for s in F5_SETS for v in ("old", "new")]])
print("\nProducts whose MODEL FAMILY changes in the BACKTEST set (weeks 1-53):")
display(f5_cmp[(f5_cmp["set"] == "backtest") & (f5_cmp["family_old"] != f5_cmp["family_new"])]
        .sort_values("weeks_before_launch", ascending=False)
        [["Product_Code", "launch_week", "weeks_before_launch", "class_old", "class_new", "family_old", "family_new"]])
print("\nNEXT: re-run cells 61, 62 (Fix 3), 63 (3b) and 64 (F4) in that order, so the new classes reach"
      " product_routing_labels.csv.")


In [ ]:
# ==================================================================
# STEP 2 — Run it on Hierarchy A's bottom level (Product_Code)
# — this is where we expect it to matter most
# ==================================================================
results_product = []
for pid, group in bottom_df.groupby("Product_Code"):
    series = group.set_index("ds")["quantity_cases"].reindex(full_calendar)
    results_product.append({"unique_id": pid, "class": classify_demand(series)})

classification_product = pd.DataFrame(results_product)
print("--- Product (SKU level) ---")
print(classification_product["class"].value_counts())
print()

classification_product.to_csv("../eabl parsed csv/classification_product.csv", index=False)

# ==================================================================
# STEP 3 — Spot-check the other three hierarchies' bottom levels too,
# since aggregation *usually* smooths out intermittency, but shouldn't
# just be assumed without checking
# ==================================================================
def classify_bottom_level(Y_df, tags, spec_last_level, full_cal):
    bottom_tag = "/".join(spec_last_level)
    bottom_ids = tags[bottom_tag]
    results = []
    for uid in bottom_ids:
        series = (
            Y_df[Y_df["unique_id"] == uid]
            .set_index("ds")["y"]
            .reindex(full_cal)
        )
        results.append({"unique_id": uid, "class": classify_demand(series)})
    return pd.DataFrame(results)

print("--- Geographic (Route level) ---")
geo_check = classify_bottom_level(Y_geo, tags_geo, spec_geo[-1], full_calendar)
print(geo_check["class"].value_counts())
print()

print("--- Channel level ---")
channel_check = classify_bottom_level(Y_channel, tags_channel, spec_channel[-1], full_calendar)
print(channel_check["class"].value_counts())
print()

print("--- Grouped (Category x Segment) ---")
grouped_check = classify_bottom_level(Y_grouped, tags_grouped, spec_grouped[-1], full_calendar)
print(grouped_check["class"].value_counts())


In [ ]:
# ==========================================================================
# STEP FIX 3: BACKTEST demand class = the SAME classify_demand rule, using ONLY weeks 1-53
#             (up to the seal, Monday 6 Apr 2026 - STEP 0 rule (a))
#
# WHY:
#   The class (smooth / erratic / intermittent / lumpy) decides each product's MODEL FAMILY
#   in F1: smooth + erratic -> standard models (Naive, SeasonalNaive, AutoETS, AutoARIMA,
#   AutoTheta, AutoCES); intermittent + lumpy -> Croston-type models (CrostonClassic, SBA,
#   TSB, ADIDA, IMAPA). The classification cell below counted all 66 weeks, so for the
#   backtest some products would be routed using sales from the sealed test weeks.
#   FPP2 3.4: "The accuracy of forecasts can only be determined by considering how well a
#   model performs on new data that were not used when fitting the model." Choosing the
#   model family is part of fitting, so it must use weeks 1-53 only.
#   The rule itself (ADI 1.32 / CV2 0.49) is Syntetos, Boylan & Croston (2005, J. Oper. Res.
#   Soc.), as already written in the classify_demand cell - it is NOT changed here.
#
# TWO CLASS SETS FROM NOW ON:
#   classification_product          -> classification cell (66 weeks) -> FINAL, final forecast only
#   classification_product_backtest -> THIS cell (53 weeks)          -> BACKTEST, CV + sealed test
#
# WHERE EACH INPUT COMES FROM (all already in memory - nothing is read from disk):
#   bottom_df      -> Prep cell 49 (gap check): weekly cases per Product_Code, built from df
#                     (drinks only after R1, cell 10); one row per product-week WITH a sale
#   full_calendar  -> Prep cell 49: the 66 Monday weeks, 7 Apr 2025 - 6 Jul 2026
#   classify_demand -> Prep cell 60: the rule (returns smooth/intermittent/erratic/lumpy/no_demand)
#   classification_product -> Prep cell 61: the 66-week (FINAL) classes, unique_id = Product_Code
#   f2_labels_backtest, F2B_CUTOFF -> Fix 1 (Prep cell 29): the 181 products known by 6 Apr 2026
#                     (unique_id path, Product_Code, SKU) and the seal date 2026-04-06
#   pd, np         -> Prep cell 0 (imports);  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing. bottom_df, full_calendar, classification_product are only read.
# WHAT IT WRITES:  no file (classification_product.csv is NOT touched; saving is F4).
#                  New table in memory: classification_product_backtest (181 rows).
# All new names start with fx3_ / FX3_ so nothing clashes with F3 (f3_) or other cells.
# ==========================================================================

FX3_FAMILY = {"smooth": "standard", "erratic": "standard",          # same split as F1's routing cell
              "intermittent": "intermittent", "lumpy": "intermittent"}

# ---- Safety checks: stop if the inputs are not the ones this cell expects ----
assert callable(classify_demand), "run the classify_demand cell (Prep cell 60) first"
assert "classification_product" in globals(), "run the classification cell (Prep cell 61) first"
assert {"Product_Code", "ds", "quantity_cases"}.issubset(bottom_df.columns), "bottom_df missing - run Prep cell 49"
assert len(full_calendar) == 66 and F2B_CUTOFF in full_calendar, "full_calendar is not the 66-week calendar - run Prep cell 49"
assert "f2_labels_backtest" in globals() and (f2_labels_backtest["label_set"] == "BACKTEST").all(), \
    "run the Fix 1 cell (Prep cell 29) first"

# ---- The same loop as the classification cell, as a function (calendar = the weeks it may see) ----
def fx3_classify_all(fx3_bd, fx3_cal):
    """fx3_bd: rows Product_Code / ds / quantity_cases; fx3_cal: the Monday weeks this run may see.
    Exactly the classification cell's steps: reindex each product on the calendar, classify_demand."""
    fx3_rows = []
    for fx3_pid, fx3_g in fx3_bd.groupby("Product_Code"):
        fx3_s = fx3_g.set_index("ds")["quantity_cases"].reindex(fx3_cal)       # weeks without a sale -> NaN
        fx3_nz = fx3_s.dropna()
        fx3_nz = fx3_nz[fx3_nz > 0]
        fx3_rows.append({"Product_Code": str(fx3_pid), "class": classify_demand(fx3_s),
                         "ADI": round(len(fx3_s) / len(fx3_nz), 2) if len(fx3_nz) else np.nan,   # for display only
                         "CV2": round((fx3_nz.std() / fx3_nz.mean()) ** 2, 2) if len(fx3_nz) > 1 else np.nan})
    return pd.DataFrame(fx3_rows)

# ---- PROOF: on all 66 weeks the function must give exactly classification_product ----
fx3_proof = fx3_classify_all(bottom_df, full_calendar).set_index("Product_Code")["class"]
fx3_ref = classification_product.assign(Product_Code=classification_product["unique_id"].astype(str)) \
                                .set_index("Product_Code")["class"]
assert fx3_proof.sort_index().equals(fx3_ref.sort_index()), \
    "the rule here does not reproduce classification_product - stop, do not use this cell"

# ---- BACKTEST run: weeks 1-53 only, products known by 6 Apr 2026 (the 181 from Fix 1) ----
fx3_cal  = full_calendar[full_calendar <= F2B_CUTOFF]                          # 53 weeks
fx3_bd   = bottom_df[bottom_df["ds"] <= F2B_CUTOFF]                            # sales up to the seal only
fx3_known = set(f2_labels_backtest["Product_Code"].astype(str))
fx3_bd   = fx3_bd[fx3_bd["Product_Code"].astype(str).isin(fx3_known)]
fx3_bt   = fx3_classify_all(fx3_bd, fx3_cal)

classification_product_backtest = (
    f2_labels_backtest[["unique_id", "Product_Code", "SKU"]].astype({"Product_Code": str})
    .merge(fx3_bt, on="Product_Code", how="left"))
classification_product_backtest["family"]    = classification_product_backtest["class"].map(FX3_FAMILY)
classification_product_backtest["cutoff"]    = F2B_CUTOFF.date()
classification_product_backtest["label_set"] = "BACKTEST"

# ---- Checks ----
assert len(fx3_cal) == 53
assert len(classification_product_backtest) == len(f2_labels_backtest) == 181
assert classification_product_backtest["Product_Code"].is_unique
assert classification_product_backtest["class"].isin(FX3_FAMILY.keys()).all(), \
    "a known product got no_demand or no class - stop and check"
assert abs(fx3_bd["quantity_cases"].sum() - f2_labels_backtest["total_cases"].sum()) < 1, \
    "bottom_df and Y_product disagree on cases up to 6 Apr 2026 - stop and check"

# ---- Report ----
print(f"BACKTEST classes: {len(classification_product_backtest)} products, {len(fx3_cal)} weeks "
      f"({fx3_cal[0].date()} to {fx3_cal[-1].date()})")
print("Proof: the same rule on all 66 weeks reproduces classification_product exactly - OK")
print("\nClass counts (BACKTEST, 53 weeks):")
print(classification_product_backtest["class"].value_counts().to_string())
print("\nModel family (BACKTEST):")
print(classification_product_backtest["family"].value_counts().to_string())

fx3_cmp = classification_product_backtest.set_index("Product_Code")[["SKU", "class", "family", "ADI", "CV2"]] \
          .rename(columns={"class": "BACKTEST (53 wks)", "family": "family BACKTEST"}) \
          .join(fx3_ref.rename("FINAL (66 wks)"))
fx3_cmp["family FINAL"] = fx3_cmp["FINAL (66 wks)"].map(FX3_FAMILY)
print("\nBACKTEST vs FINAL class (rows = 53 weeks, columns = 66 weeks):")
print(pd.crosstab(fx3_cmp["BACKTEST (53 wks)"], fx3_cmp["FINAL (66 wks)"]).to_string())

fx3_changed = fx3_cmp[fx3_cmp["BACKTEST (53 wks)"] != fx3_cmp["FINAL (66 wks)"]]
fx3_fam_changed = fx3_cmp[fx3_cmp["family BACKTEST"] != fx3_cmp["family FINAL"]]
print(f"\nProducts whose class changes once the test weeks are seen: {len(fx3_changed)}")
print(f"... of which the MODEL FAMILY changes (standard <-> intermittent): {len(fx3_fam_changed)}")
display(fx3_fam_changed[["SKU", "BACKTEST (53 wks)", "FINAL (66 wks)", "family BACKTEST", "family FINAL", "ADI", "CV2"]])
print(f"\nNot classified for the backtest (first sale after 6 Apr 2026): "
      f"{len(fx3_ref) - len(classification_product_backtest)} products (they go to the judgement lane, F3 'too new')")


In [ ]:
# ==========================================================================
# STEP 3b: CV-ROUND labels and types - the SAME F2 rules and the SAME class rule,
#          made separately for each cross-validation round, using only the weeks
#          that round is allowed to learn from
#
# WHY:
#   CV round 1 learns weeks 1-27 and predicts weeks 28-40; round 2 learns weeks 1-40 and
#   predicts weeks 41-53 (STEP 0 rule (c): h = 13, n_windows = 2, step_size = 13).
#   Fix 1 and Fix 3 made labels and types at week 53 - right for the sealed TEST, but they
#   have seen weeks 28-53, which are the answers of the two CV rounds.
#   FPP2 3.4 (time series cross-validation): "The corresponding training set consists only of
#   observations that occurred prior to the observation that forms the test set. Thus, no
#   future observations can be used in constructing the forecast."
#   Each round's routing (ZERO lane or a model, standard or intermittent family) is part of
#   constructing its forecast, so each round gets labels and types from its own weeks.
#
# FOUR LABEL / TYPE SETS FROM NOW ON:
#   f2_labels_cv1                                   -> weeks 1-27 -> CV round 1   (THIS cell)
#   f2_labels_cv2                                   -> weeks 1-40 -> CV round 2   (THIS cell)
#   f2_labels_backtest + classification_product_backtest -> weeks 1-53 -> sealed TEST (Fix 1, Fix 3)
#   f2_labels + classification_product              -> weeks 1-66 -> FINAL forecast (F2, cell 61)
#
# WHERE EACH INPUT COMES FROM (all already in memory - nothing is read from disk):
#   f2b_label_products, f2b_all, f2b_weeks_all -> Fix 1 (Prep cell 29): the F2 rules as a function,
#                     the product-level rows of Y_product (66 weeks) and the 66 Monday weeks
#   f2_labels_backtest, F2B_CUTOFF             -> Fix 1 (Prep cell 29): week-53 labels and the seal
#   f3_cv1_end, f3_cv2_end                     -> F3 (Prep cell 30): 27 and 40 = last week each round learns
#   fx3_classify_all, FX3_FAMILY               -> Fix 3 (Prep cell 62): the class rule as a function,
#                                                 and class -> family (standard / intermittent)
#   classification_product_backtest            -> Fix 3 (Prep cell 62): week-53 types
#   bottom_df, full_calendar                   -> Prep cell 49: weekly cases per Product_Code, 66 weeks
#   pd, np -> Prep cell 0 (imports);  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing. Every input above is only read.
# WHAT IT WRITES:  no file (saving is F4). New tables in memory: f2_labels_cv1, f2_labels_cv2.
# All new names start with cvr_ / CVR_ so nothing clashes with other cells.
# ==========================================================================

# ---- Safety checks: stop if the inputs are not the ones this cell expects ----
assert callable(f2b_label_products) and "f2_labels_backtest" in globals(), "run the Fix 1 cell (Prep cell 29) first"
assert "f3_cv1_end" in globals() and (f3_cv1_end, f3_cv2_end) == (27, 40), "run the F3 cell (Prep cell 30) first"
assert callable(fx3_classify_all) and "classification_product_backtest" in globals(), "run the Fix 3 cell (Prep cell 62) first"
assert len(f2b_weeks_all) == 66 and len(full_calendar) == 66, "the 66-week calendars are missing - re-run Prep from the top"

# ---- PROOF: the two functions, at week 53, must give exactly the Fix 1 and Fix 3 results ----
cvr_bt_lab = f2b_label_products(
    f2b_all[(f2b_all["ds"] <= F2B_CUTOFF) & f2b_all["unique_id"].isin(f2_labels_backtest["unique_id"])],
    f2b_weeks_all[f2b_weeks_all <= F2B_CUTOFF]).set_index("unique_id")["label"]
assert cvr_bt_lab.sort_index().equals(f2_labels_backtest.set_index("unique_id")["label"].sort_index()), \
    "f2b_label_products no longer gives the Fix 1 labels - re-run Fix 1"
cvr_bt_cls = fx3_classify_all(
    bottom_df[(bottom_df["ds"] <= F2B_CUTOFF)
              & bottom_df["Product_Code"].astype(str).isin(classification_product_backtest["Product_Code"])],
    full_calendar[full_calendar <= F2B_CUTOFF]).set_index("Product_Code")["class"]
assert cvr_bt_cls.sort_index().equals(classification_product_backtest.set_index("Product_Code")["class"].sort_index()), \
    "fx3_classify_all no longer gives the Fix 3 classes - re-run Fix 3"

# ---- One run per CV round: cut at that round's last learning week, apply the same two rules ----
def cvr_make_round(cvr_end_week, cvr_name):
    """cvr_end_week: last week (1-66) this round may learn from; cvr_name: 'CV1' or 'CV2'."""
    cvr_cut   = f2b_weeks_all[cvr_end_week - 1]                                  # Monday of that week
    cvr_weeks = f2b_weeks_all[f2b_weeks_all <= cvr_cut]
    # labels - same F2 rules, products with a sale by the cut-off only
    cvr_rows  = f2b_all[f2b_all["ds"] <= cvr_cut]
    cvr_known = cvr_rows.groupby("unique_id")["y"].sum().loc[lambda s: s > 0].index
    cvr_lab   = f2b_label_products(cvr_rows[cvr_rows["unique_id"].isin(cvr_known)], cvr_weeks)
    # types - same class rule, same products
    cvr_codes = set(cvr_lab["Product_Code"])
    cvr_bd    = bottom_df[(bottom_df["ds"] <= cvr_cut) & bottom_df["Product_Code"].astype(str).isin(cvr_codes)]
    cvr_cls   = fx3_classify_all(cvr_bd, full_calendar[full_calendar <= cvr_cut])
    cvr_out   = cvr_lab.merge(cvr_cls, on="Product_Code", how="left")
    cvr_out["family"]    = cvr_out["class"].map(FX3_FAMILY)
    cvr_out["cutoff"]    = cvr_cut.date()
    cvr_out["label_set"] = cvr_name
    # checks for this round
    assert len(cvr_weeks) == cvr_end_week and cvr_out["unique_id"].is_unique
    assert cvr_out["class"].isin(FX3_FAMILY.keys()).all(), f"{cvr_name}: a product has no class - stop and check"
    assert abs(cvr_out["total_cases"].sum() - cvr_rows["y"].sum()) < 1, f"{cvr_name}: cases lost - stop and check"
    assert abs(cvr_bd["quantity_cases"].sum() - cvr_rows["y"].sum()) < 1, f"{cvr_name}: bottom_df and Y_product disagree"
    assert pd.to_datetime(cvr_out["last_sale_week"]).max() <= cvr_cut, f"{cvr_name}: a later week was used - stop"
    return cvr_out

f2_labels_cv1 = cvr_make_round(f3_cv1_end, "CV1")     # weeks 1-27, cut-off Mon 6 Oct 2025
f2_labels_cv2 = cvr_make_round(f3_cv2_end, "CV2")     # weeks 1-40, cut-off Mon 5 Jan 2026
assert str(f2_labels_cv1["cutoff"].iloc[0]) == "2025-10-06" and str(f2_labels_cv2["cutoff"].iloc[0]) == "2026-01-05"

# ---- Report: each round compared with the week-53 (BACKTEST) labels and types ----
cvr_ref = (f2_labels_backtest.set_index("unique_id")[["label"]]
           .join(classification_product_backtest.set_index("unique_id")[["class", "family"]]))
print("Proof: Fix 1 and Fix 3 functions reproduce the week-53 labels and types exactly - OK")
for cvr_tab in [f2_labels_cv1, f2_labels_cv2]:
    cvr_name = cvr_tab["label_set"].iloc[0]
    cvr_j = cvr_tab.set_index("unique_id")[["SKU", "label", "class", "family"]].join(cvr_ref, rsuffix=" (wk53)")
    cvr_lab_diff = cvr_j[cvr_j["label"] != cvr_j["label (wk53)"]]
    cvr_cls_diff = cvr_j[cvr_j["class"] != cvr_j["class (wk53)"]]
    cvr_fam_diff = cvr_j[cvr_j["family"] != cvr_j["family (wk53)"]]
    print(f"\n==================== {cvr_name}: learns weeks 1-{len(f2b_weeks_all[f2b_weeks_all <= pd.Timestamp(cvr_tab['cutoff'].iloc[0])])} "
          f"(to {cvr_tab['cutoff'].iloc[0]}) | products known: {len(cvr_tab)} ====================")
    print("Labels:", cvr_tab["label"].value_counts().to_dict())
    print("Types: ", cvr_tab["class"].value_counts().to_dict(), "| family:", cvr_tab["family"].value_counts().to_dict())
    print(f"Differ from week 53 -> labels: {len(cvr_lab_diff)} | types: {len(cvr_cls_diff)} | model family: {len(cvr_fam_diff)}")
    print("\nLabel here vs week 53:")
    print(pd.crosstab(cvr_j["label"], cvr_j["label (wk53)"]).to_string())
    print("\nProducts whose label OR model family differs from week 53:")
    display(cvr_j[(cvr_j["label"] != cvr_j["label (wk53)"]) | (cvr_j["family"] != cvr_j["family (wk53)"])]
            [["SKU", "label", "label (wk53)", "class", "class (wk53)", "family", "family (wk53)"]]
            .sort_values(["label", "family"]))


In [ ]:
# ==========================================================================
# STEP F4: SAVE all labels, types and history groups to ONE new file for F1
#          ../eabl parsed csv/product_routing_labels.csv   (one row per product, 188 rows)
#
# WHY:
#   Everything from F2, Fix 1, F3, Fix 3 and 3b lives only in this notebook's memory. F1 is a
#   separate notebook and cannot see it, so it must be saved to a file F1 can read (Step 7).
#   FPP2 4.2 "Document and justify": "Formalising and documenting the decision rules and
#   assumptions implemented in the systematic approach can promote consistency, as the same
#   rules can be implemented repeatedly."  -> one documented file, the same for every F1 run.
#   Leak row 6 (wrong file): every column says WHICH weeks it saw (_cv1 / _cv2 / _backtest /
#   _final + 4 cut-off columns), so F1 can check it uses the right set for each forecast.
#   Leak row 7 (July 2026 master): master Status is saved ONLY as master_status_INFO_ONLY -
#   a question for EABL (Step 5), never used to label or route.
#
# WHICH SET IS FOR WHAT (STEP 0 rules):
#   *_cv1      weeks 1-27 (cut-off 2025-10-06) -> CV round 1
#   *_cv2      weeks 1-40 (cut-off 2026-01-05) -> CV round 2
#   *_backtest weeks 1-53 (cut-off 2026-04-06) -> the sealed test (weeks 54-66)
#   *_final    weeks 1-66 (cut-off 2026-07-06) -> the final forecast only (weeks 67-79)
#   "not known yet" = the product had no sale by that cut-off
#
# WHERE EACH INPUT COMES FROM (all already in memory):
#   f2_labels                       -> F2 (Prep cell 28): 188 products, FINAL label, master_status
#   f2_labels_backtest, F2B_CUTOFF  -> Fix 1 (Prep cell 29): 181 products, BACKTEST label; the seal
#   f3_history                      -> F3 (Prep cell 30): 188 products, history group, weeks_in_train
#   classification_product          -> Prep cell 61: 188 FINAL classes (unique_id = Product_Code)
#   classification_product_backtest -> Fix 3 (Prep cell 62): 181 BACKTEST classes + family
#   FX3_FAMILY                      -> Fix 3 (Prep cell 62): class -> family (standard / intermittent)
#   f2_labels_cv1, f2_labels_cv2    -> 3b (Prep cell 63): 171 / 175 products, label + class + family
#   f2b_weeks_all                   -> Fix 1 (Prep cell 29): the 66 Monday weeks (last = final cut-off)
#   pd, os                          -> pandas (Prep cell 0); os is Python's built-in file helper
#
# WHAT IT CHANGES: nothing in memory - every input above is only read.
# WHAT IT WRITES:  ONE NEW file, ../eabl parsed csv/product_routing_labels.csv.
#                  It does NOT touch Y_product.csv, S_product.csv, classification_product.csv
#                  or any other file. If a file with this name already exists and was NOT made
#                  by this cell (different columns), the cell stops and writes nothing.
# All new names start with f4_ / F4_ so nothing clashes with other cells.
# ==========================================================================
import os

F4_PATH = "../eabl parsed csv/product_routing_labels.csv"
F4_NOT_KNOWN = "not known yet"

# ---- Safety checks: stop (and write nothing) if any input is missing or not the expected table ----
assert "f2_labels" in globals() and len(f2_labels) == 188, "run the F2 cell (Prep cell 28) first"
assert (f2_labels_backtest["label_set"] == "BACKTEST").all() and len(f2_labels_backtest) == 181, "run Fix 1 (Prep cell 29)"
assert "f3_history" in globals() and len(f3_history) == 188, "run F3 (Prep cell 30) first"
assert "classification_product" in globals() and len(classification_product) == 188, "run Prep cell 61 first"
assert (classification_product_backtest["label_set"] == "BACKTEST").all() and len(classification_product_backtest) == 181, \
    "run Fix 3 (Prep cell 62) first"
assert (f2_labels_cv1["label_set"] == "CV1").all() and (f2_labels_cv2["label_set"] == "CV2").all(), "run 3b (Prep cell 63) first"

# ---- The cut-off dates, read from the tables themselves (not typed again) ----
f4_cut = {"cv1": f2_labels_cv1["cutoff"].iloc[0], "cv2": f2_labels_cv2["cutoff"].iloc[0],
          "backtest": f2_labels_backtest["cutoff"].iloc[0], "final": f2b_weeks_all[-1].date()}
assert [str(f4_cut[k]) for k in ["cv1", "cv2", "backtest", "final"]] == \
       ["2025-10-06", "2026-01-05", "2026-04-06", "2026-07-06"], f"unexpected cut-offs: {f4_cut}"

# ---- Build ONE row per product (all 188), starting from the F2 product list ----
f4_out = f2_labels[["unique_id", "Product_Code", "SKU", "first_sale_week"]].copy()
f4_out["Product_Code"] = f4_out["Product_Code"].astype(str)

# F3 history group (leak-free: first sale date only)
f4_out = f4_out.merge(f3_history[["unique_id", "history", "weeks_in_train", "cv_rounds_usable"]],
                      on="unique_id", how="left")

# labels: FINAL (66 wks), BACKTEST (53), CV2 (40), CV1 (27)
f4_out["label_final"] = f4_out["unique_id"].map(f2_labels.set_index("unique_id")["label"])
for f4_name, f4_tab in [("backtest", f2_labels_backtest), ("cv2", f2_labels_cv2), ("cv1", f2_labels_cv1)]:
    f4_out[f"label_{f4_name}"] = f4_out["unique_id"].map(f4_tab.set_index("unique_id")["label"]).fillna(F4_NOT_KNOWN)

# types (class) and model family: FINAL, BACKTEST, CV2, CV1
f4_out["class_final"] = f4_out["Product_Code"].map(
    classification_product.assign(pc=classification_product["unique_id"].astype(str)).set_index("pc")["class"])
for f4_name, f4_tab in [("backtest", classification_product_backtest), ("cv2", f2_labels_cv2), ("cv1", f2_labels_cv1)]:
    f4_out[f"class_{f4_name}"] = f4_out["unique_id"].map(f4_tab.set_index("unique_id")["class"]).fillna(F4_NOT_KNOWN)
for f4_name in ["final", "backtest", "cv2", "cv1"]:
    f4_out[f"family_{f4_name}"] = f4_out[f"class_{f4_name}"].map(FX3_FAMILY).fillna(F4_NOT_KNOWN)

# cut-off stamps (which weeks each set saw) and master Status as INFO ONLY
for f4_name in ["cv1", "cv2", "backtest", "final"]:
    f4_out[f"cutoff_{f4_name}"] = f4_cut[f4_name]
f4_out["master_status_INFO_ONLY"] = f4_out["unique_id"].map(f2_labels.set_index("unique_id")["master_status"])

# ---- Checks before writing ----
assert len(f4_out) == 188 and f4_out["unique_id"].is_unique
assert f4_out.drop(columns=["master_status_INFO_ONLY"]).notna().all().all(), "a column has gaps - stop and check"
assert (f4_out["label_backtest"] != F4_NOT_KNOWN).sum() == 181 and (f4_out["label_cv2"] != F4_NOT_KNOWN).sum() == 175 \
       and (f4_out["label_cv1"] != F4_NOT_KNOWN).sum() == 171, "products per set do not match Fix 1 / 3b"
assert ((f4_out["history"] == "too new") == (f4_out["label_backtest"] == F4_NOT_KNOWN)).all(), \
    "F3 'too new' and Fix 1 'not known yet' disagree - stop and check"
assert ((f4_out["label_backtest"] == F4_NOT_KNOWN) == (f4_out["class_backtest"] == F4_NOT_KNOWN)).all()
assert f4_out["class_final"].isin(FX3_FAMILY.keys()).all(), "a product has no FINAL class - stop and check"

# ---- Never overwrite a file this cell did not make ----
if os.path.exists(F4_PATH):
    f4_old_cols = list(pd.read_csv(F4_PATH, nrows=0).columns)
    assert f4_old_cols == list(f4_out.columns), \
        f"{F4_PATH} already exists with different columns - not made by F4. Nothing written; rename or check it."
    print(f"Note: {F4_PATH} already existed (made by an earlier F4 run) - it is replaced with this run's version.")

# ---- Write, then read back and compare (proves the file holds exactly the table) ----
f4_out.to_csv(F4_PATH, index=False)
f4_back = pd.read_csv(F4_PATH, dtype=str)
assert f4_back.shape == f4_out.shape and (f4_back.fillna("") == f4_out.astype(str).replace("nan", "")).all().all(), \
    "the saved file does not match the table - stop and check"

# ---- Report ----
print(f"Saved {F4_PATH}: {f4_out.shape[0]} products x {f4_out.shape[1]} columns  (read back and checked - OK)")
print("\nCut-offs:", {k: str(v) for k, v in f4_cut.items()})
print("\nColumns:", list(f4_out.columns))
print("\nProducts with a label in each set:",
      {s: int((f4_out[f"label_{s}"] != F4_NOT_KNOWN).sum()) for s in ["cv1", "cv2", "backtest", "final"]})
print("\nHistory groups:", f4_out["history"].value_counts().to_dict())
print("\nFamily per set:")
print(pd.DataFrame({s: f4_out[f"family_{s}"].value_counts() for s in ["cv1", "cv2", "backtest", "final"]})
      .fillna(0).astype(int).to_string())
print("\nExample rows:")
display(f4_out[f4_out["Product_Code"].isin(["741710", "624133", "803864", "799700"])]
        [["SKU", "history", "label_cv1", "label_cv2", "label_backtest", "label_final",
          "family_cv1", "family_backtest", "family_final", "master_status_INFO_ONLY"]])

In [ ]:
# ==========================================================================
# STEP 6 - PART 1 (F6 + F7): HOLIDAY CHECKS on weeks 1-53 only - FLAG ONLY, nothing is changed
#   F6: do empty / short / low weeks fall on public-holiday weeks?
#   F7: do sales jump in the week AFTER a public holiday?
#
# WHY:
#   FPP2 12.9: "Missing data can arise for many reasons, and it is worth considering whether the
#     missingness will induce bias in the forecasting model. For example, suppose we are studying
#     sales data for a store, and missing values occur on public holidays when the store is closed.
#     The following day may have increased sales as a result."  -> F6 (the dip) and F7 (the bounce).
#   FPP2 5.4: holidays can be handled with "a dummy variable"; "Easter differs from most holidays
#     because it is not held on the same date each year, and its effect can last for several days."
#     -> this cell only MEASURES the effect; whether F1 needs a holiday dummy is decided at Step 7.
#
# LEAK RULES (STEP 0 + Step 6):
#   - only weeks 1-53 (up to the seal, Mon 6 Apr 2026) are looked at - weeks 54-66 are not used
#   - "normal" for a week = the median of the 8 weeks BEFORE it that are not holiday or
#     after-holiday weeks (earlier dates only; never later weeks). Needs at least 4 such weeks,
#     otherwise "too early to judge". (D2's cases_vs_median_week uses all 66 weeks, so it is NOT used.)
#   - flag only: no sales value is changed, no row is removed, no file is written here
#     (the flags are saved to a NEW file in Step 6 part 3, F9).
#
# SETTINGS (our documented choices, not FPP2 numbers):
#   F6_LOOKBACK = 8 weeks of history for "normal"; F6_LOW = 0.85 / F6_HIGH = 1.15 -> a week is
#   "low" if below 85% of normal and "high" if above 115% of normal.
#
# WHERE EACH INPUT COMES FROM (all already in memory - nothing is read from disk):
#   d2_weeks   -> D2 (Prep cell 27): one row per week (66) - week_start, holidays_mon_sat,
#                 holiday_names, near_empty_days, near_empty_list, cases (all drinks)
#   f2b_all    -> Fix 1 (Prep cell 29): product-level rows of Y_product (unique_id, ds, y), 66 weeks
#   F2B_CUTOFF -> Fix 1 (Prep cell 29): Timestamp 2026-04-06, the seal (last week = week 53)
#   pd, np     -> Prep cell 0 (imports);  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing. d2_weeks and f2b_all are only read.
# WHAT IT WRITES:  no file. New table in memory: f6_week_flags (53 rows, one per week).
# All new names start with f6_ / F6_ so nothing clashes with other cells.
# ==========================================================================

F6_LOOKBACK, F6_MIN_BASE = 8, 4          # weeks of earlier history for "normal"; minimum needed
F6_LOW, F6_HIGH = 0.85, 1.15             # below 85% of normal = low, above 115% = high
F6_SERIES = ["Total", "Beer", "RTD", "Spirits"]

# ---- Safety checks: stop if the inputs are not the ones this cell expects ----
assert {"week_start", "holidays_mon_sat", "holiday_names", "near_empty_days", "near_empty_list", "cases"} \
    .issubset(d2_weeks.columns) and len(d2_weeks) == 66, "run the D2 cell (Prep cell 27) first"
assert "F2B_CUTOFF" in globals() and {"unique_id", "ds", "y"}.issubset(f2b_all.columns), "run Fix 1 (Prep cell 29) first"

# ---- Weeks 1-53 only, with holiday and after-holiday markers ----
f6_w = d2_weeks[pd.to_datetime(d2_weeks["week_start"]) <= F2B_CUTOFF].copy().reset_index(drop=True)
f6_w["week_start"] = pd.to_datetime(f6_w["week_start"])
assert len(f6_w) == 53 and f6_w["week_start"].max() == F2B_CUTOFF, "expected exactly weeks 1-53"
f6_w.insert(1, "week_no", range(1, 54))
f6_w["is_holiday_week"] = f6_w["holidays_mon_sat"] > 0
f6_w["after_holiday_week"] = f6_w["is_holiday_week"].shift(1, fill_value=False)
f6_w["holiday_names"] = f6_w["holiday_names"].fillna("")
f6_w["near_empty_list"] = f6_w["near_empty_list"].fillna("")
# a near-empty day that IS the holiday (e.g. "Thu 01 Jan" appears in both lists)
f6_w["near_empty_on_holiday"] = [any(d.strip() and d.strip() in h for d in ne.split(","))
                                 for ne, h in zip(f6_w["near_empty_list"], f6_w["holiday_names"])]

# ---- Weekly cases: total (from D2) and per Category2 (from the product rows), weeks 1-53 ----
f6_cat = (f2b_all[f2b_all["ds"] <= F2B_CUTOFF]
          .assign(category=lambda t: t["unique_id"].str.split("/").str[0])
          .pivot_table(index="ds", columns="category", values="y", aggfunc="sum"))
f6_w = f6_w.join(f6_cat, on="week_start")
f6_w["Total"] = f6_w["cases"]
assert set(["Beer", "RTD", "Spirits"]).issubset(f6_w.columns), "expected categories Beer / RTD / Spirits"
assert (abs(f6_w[["Beer", "RTD", "Spirits"]].sum(axis=1) - f6_w["Total"]) < 1).all(), \
    "category totals do not add up to D2's weekly cases - stop and check"

# ---- "Normal" for each week = median of the earlier normal weeks (earlier dates only) ----
f6_normal_mask = ~(f6_w["is_holiday_week"] | f6_w["after_holiday_week"])
for f6_s in F6_SERIES:
    f6_base = []
    for f6_t in range(len(f6_w)):
        f6_prev = f6_w.loc[max(0, f6_t - F6_LOOKBACK):f6_t - 1]                  # weeks BEFORE week t only
        f6_prev = f6_prev.loc[f6_normal_mask.loc[f6_prev.index], f6_s]
        f6_base.append(f6_prev.median() if len(f6_prev) >= F6_MIN_BASE else np.nan)
    f6_w[f"{f6_s}_normal"] = f6_base
    f6_w[f"{f6_s}_vs_normal"] = (f6_w[f6_s] / f6_w[f"{f6_s}_normal"]).round(2)
f6_r = f6_w["Total_vs_normal"]
f6_w["flag_total"] = np.select([f6_r.isna(), f6_r < F6_LOW, f6_r > F6_HIGH], ["too early", "low", "high"], "normal")

f6_week_flags = f6_w[["week_start", "week_no", "is_holiday_week", "holiday_names", "after_holiday_week",
                      "near_empty_days", "near_empty_list", "near_empty_on_holiday"]
                     + [c for s in F6_SERIES for c in (s, f"{s}_normal", f"{s}_vs_normal")] + ["flag_total"]].copy()

# ---- Checks ----
assert len(f6_week_flags) == 53 and f6_week_flags["week_start"].max() <= F2B_CUTOFF
assert abs(f6_week_flags["Total"].sum() - f2b_all.loc[f2b_all["ds"] <= F2B_CUTOFF, "y"].sum()) < 1

# ---- Report F6: holiday weeks and low weeks ----
f6_show = ["week_start", "week_no", "holiday_names", "near_empty_list", "near_empty_on_holiday",
           "Total_vs_normal", "Beer_vs_normal", "RTD_vs_normal", "Spirits_vs_normal", "flag_total"]
f6_hol = f6_week_flags[f6_week_flags["is_holiday_week"]]
print(f"F6 - weeks 1-53 ({f6_week_flags['week_start'].min().date()} to {f6_week_flags['week_start'].max().date()}): "
      f"{len(f6_hol)} holiday weeks, {int(f6_week_flags['near_empty_days'].sum())} near-empty days")
print("'x_vs_normal' = that week's cases / median of the 8 earlier normal weeks (1.00 = normal)")
display(f6_hol[f6_show])
f6_low = f6_week_flags[f6_week_flags["flag_total"] == "low"]
print(f"\nLOW weeks (total below {F6_LOW:.0%} of normal): {len(f6_low)} | of which holiday weeks: "
      f"{int(f6_low['is_holiday_week'].sum())} | after-holiday weeks: {int(f6_low['after_holiday_week'].sum())}")
display(f6_low[f6_show])
print("\nNear-empty days that fall ON the holiday itself:",
      f6_week_flags.loc[f6_week_flags["near_empty_on_holiday"], "near_empty_list"].tolist())

# ---- Report F7: the week after a holiday ----
f6_after = f6_week_flags[f6_week_flags["after_holiday_week"]]
f6_plain = f6_week_flags[~(f6_week_flags["is_holiday_week"] | f6_week_flags["after_holiday_week"])]
print(f"\nF7 - weeks right AFTER a holiday week: {len(f6_after)}")
display(f6_after[f6_show])
print("\nMedian 'vs normal' (Total / Beer / RTD / Spirits):")
print(pd.DataFrame({"holiday weeks": f6_hol[[f"{s}_vs_normal" for s in F6_SERIES]].median(),
                    "after-holiday weeks": f6_after[[f"{s}_vs_normal" for s in F6_SERIES]].median(),
                    "other weeks": f6_plain[[f"{s}_vs_normal" for s in F6_SERIES]].median()})
      .rename(index=lambda c: c.replace("_vs_normal", "")).round(2).to_string())

In [ ]:
# ==========================================================================
# STEP 6 - PART 2 (F8): SPIKE AND DROP SCREEN for every product, weeks 1-53 only - FLAG ONLY
#   SPIKE = a week far ABOVE what that product usually sells (e.g. a bulk order, a promotion)
#   DROP  = a week far BELOW normal for a product that sells almost every week
#           (a possible stock-out - to check with EABL; nothing is filled in)
#
# WHY:
#   FPP2 12.9: "Outliers are observations that are very different from the majority of the
#     observations in the time series. They may be errors, or they may simply be unusual."
#     "Simply replacing outliers without thinking about why they have occurred is a dangerous
#     practice. They may provide useful information about the process that produced the data"
#     -> we only FIND and LABEL them here; any change is a later, approved decision.
#   FPP2 5.4: a one-off event can be handled with a "spike" variable (a dummy that is 1 in that
#     week) instead of deleting the week -> the flags below are what such a variable would use.
#   FPP2 12.9 (missing values): "it is worth considering whether the missingness will induce
#     bias" -> a regular seller suddenly at ~0 may be a stock-out, not low demand.
#   Hierarchy (FPP2 10.3): product weeks add up into brand / category totals, so an odd product
#     week also shows up higher in the hierarchy.
#
# LEAK RULES (STEP 0 + Step 6):
#   - only weeks 1-53 (up to the seal, Mon 6 Apr 2026); weeks 54-66 are not used
#   - EARLIER DATES ONLY: each week is compared with the 13 weeks BEFORE it (never later weeks).
#     This replaces two older checks that looked at all dates: the Sunday-spike rule
#     (Cleaning cells 40-41, leak C) and the stock-out guess that looked 4 weeks AHEAD
#     (Prep cells 50 and 56, leak J). Because every flag only looks back, the same flags are
#     also valid for CV round 1 (weeks <= 27) and round 2 (weeks <= 40).
#   - flag only: nothing is changed or removed; no file is written here (saving is Part 3, F9)
#
# THE RULES (our documented choices, not FPP2 numbers):
#   history  = the 13 weeks before the week (one quarter), counted from the product's first
#              sale; at least 8 weeks needed, otherwise "too early to judge"
#   SPIKE    = sold more than 3 x its usual NON-ZERO week (median of earlier selling weeks,
#              needs at least 3) AND above median + 3 x 1.4826 x MAD of the earlier weeks
#              (the same robust rule as Cleaning cell 40, now using earlier weeks only)
#   DROP     = the product sold in at least 75% of the earlier weeks, and this week is
#              below 25% of its usual week (median of the earlier weeks)
#   SIZE     = tiny wobbles are ignored: a spike must add at least 1 case over the usual selling
#              week, and a drop is only checked for products whose usual week is at least 1 case
#
# WHERE EACH INPUT COMES FROM:
#   f2b_all            -> Fix 1 (Prep cell 29): product rows of Y_product (unique_id, ds, y), zero-filled
#                         from each product's first sale (Prep cell 23), 66 weeks
#   F2B_CUTOFF         -> Fix 1 (Prep cell 29): 2026-04-06, the seal
#   f2_labels_backtest -> Fix 1 (Prep cell 29): SKU name and BACKTEST label per product
#   classification_product_backtest -> Fix 3 (Prep cell 62): BACKTEST type and family per product
#   f6_week_flags      -> Step 6 Part 1 (Prep cell 65): holiday / after-holiday markers per week
#   ../eabl parsed csv/known_spikes.csv -> saved by Cleaning cell 41 (8 Sunday-spike product-weeks,
#                         week of 8 Dec 2025) - READ ONLY, to check whether F8 finds them too
#   pd, np -> Prep cell 0 (imports);  display -> Jupyter built-in
#
# WHAT IT CHANGES: nothing. Every input above is only read (known_spikes.csv is read, not written).
# WHAT IT WRITES:  no file. New tables in memory: f8_product_week (every product-week, weeks 1-53,
#                  with its flag) and f8_flags (only the flagged ones).
# All new names start with f8_ / F8_ so nothing clashes with other cells.
# ==========================================================================

F8_HIST, F8_MIN_HIST, F8_MIN_NZ = 13, 8, 3   # weeks looked back; minimum weeks; minimum selling weeks
F8_SPIKE_X, F8_MAD_K = 3.0, 3.0              # spike: > 3 x usual selling week AND > median + 3 x MAD
F8_REGULAR, F8_DROP_X = 0.75, 0.25           # drop: sells in >= 75% of weeks AND this week < 25% of usual
F8_MIN_CASES = 1.0                           # size filter: a spike adds >= 1 case over usual; a drop needs a usual week >= 1 case
F8_KNOWN_SPIKES = "../eabl parsed csv/known_spikes.csv"

# ---- Safety checks: stop if the inputs are not the ones this cell expects ----
assert {"unique_id", "ds", "y"}.issubset(f2b_all.columns) and "F2B_CUTOFF" in globals(), "run Fix 1 (Prep cell 29) first"
assert (f2_labels_backtest["label_set"] == "BACKTEST").all(), "run Fix 1 (Prep cell 29) first"
assert (classification_product_backtest["label_set"] == "BACKTEST").all(), "run Fix 3 (Prep cell 62) first"
assert "f6_week_flags" in globals() and len(f6_week_flags) == 53, "run Step 6 Part 1 (Prep cell 65) first"

# ---- Product rows, weeks 1-53 only, products known by 6 Apr 2026 (the 181 from Fix 1) ----
f8_rows = f2b_all[(f2b_all["ds"] <= F2B_CUTOFF) & f2b_all["unique_id"].isin(f2_labels_backtest["unique_id"])] \
          .sort_values(["unique_id", "ds"]).reset_index(drop=True)
assert f8_rows["ds"].max() == F2B_CUTOFF and f8_rows["unique_id"].nunique() == 181

# ---- Judge every product-week against ITS OWN earlier weeks only ----
f8_out = []
for f8_uid, f8_g in f8_rows.groupby("unique_id", sort=False):
    f8_y = f8_g["y"].to_numpy()
    for f8_t in range(len(f8_y)):
        f8_prev = f8_y[max(0, f8_t - F8_HIST):f8_t]                       # earlier weeks only (since first sale)
        f8_rec = {"unique_id": f8_uid, "ds": f8_g["ds"].iloc[f8_t], "y": f8_y[f8_t],
                  "usual_week": np.nan, "usual_selling_week": np.nan, "x_usual": np.nan, "flag": "too early"}
        if len(f8_prev) >= F8_MIN_HIST:
            f8_med = np.median(f8_prev)
            f8_mad = 1.4826 * np.median(np.abs(f8_prev - f8_med))
            f8_nz = f8_prev[f8_prev > 0]
            f8_med_nz = np.median(f8_nz) if len(f8_nz) >= F8_MIN_NZ else np.nan
            f8_rec.update(usual_week=round(f8_med, 2), usual_selling_week=round(f8_med_nz, 2) if f8_med_nz == f8_med_nz else np.nan)
            f8_spike = (f8_med_nz == f8_med_nz) and f8_y[f8_t] > F8_SPIKE_X * f8_med_nz and f8_y[f8_t] > f8_med + F8_MAD_K * f8_mad \
                       and f8_y[f8_t] - f8_med_nz >= F8_MIN_CASES
            f8_drop = (f8_prev > 0).mean() >= F8_REGULAR and f8_med >= F8_MIN_CASES and f8_y[f8_t] < F8_DROP_X * f8_med
            f8_ref = f8_med_nz if f8_spike else f8_med
            f8_rec["x_usual"] = round(f8_y[f8_t] / f8_ref, 2) if f8_ref and f8_ref == f8_ref else np.nan
            f8_rec["flag"] = "spike" if f8_spike else ("drop" if f8_drop else "normal")
        f8_out.append(f8_rec)
f8_product_week = pd.DataFrame(f8_out)

# ---- Add week markers, product names, labels and types ----
f8_product_week = (f8_product_week
    .merge(f6_week_flags[["week_start", "week_no", "is_holiday_week", "after_holiday_week", "flag_total"]]
           .rename(columns={"week_start": "ds", "flag_total": "company_week_flag"}), on="ds", how="left")
    .merge(f2_labels_backtest[["unique_id", "Product_Code", "SKU", "label"]].rename(columns={"label": "label_backtest"}),
           on="unique_id", how="left")
    .merge(classification_product_backtest[["unique_id", "class", "family"]]
           .rename(columns={"class": "class_backtest", "family": "family_backtest"}), on="unique_id", how="left"))

# ---- Were the Sunday spikes from Cleaning (known_spikes.csv) found too? (read-only) ----
f8_ks = pd.read_csv(F8_KNOWN_SPIKES, usecols=["Product_Code", "week_start", "Effect of Sunday"])
f8_ks["Product_Code"] = f8_ks["Product_Code"].astype(str)
f8_ks["ds"] = pd.to_datetime(f8_ks["week_start"])
f8_product_week = f8_product_week.merge(f8_ks[["Product_Code", "ds", "Effect of Sunday"]]
                                        .rename(columns={"Effect of Sunday": "known_spike_note"}),
                                        on=["Product_Code", "ds"], how="left")
f8_flags = f8_product_week[f8_product_week["flag"].isin(["spike", "drop"])].copy()

# ---- Checks ----
assert len(f8_product_week) == len(f8_rows), "rows lost while screening - stop and check"
assert abs(f8_product_week["y"].sum() - f8_rows["y"].sum()) < 1, "cases changed - stop and check"
assert f8_product_week["ds"].max() <= F2B_CUTOFF and f8_product_week["week_no"].notna().all()

# ---- Report ----
print(f"F8 - {f8_product_week['unique_id'].nunique()} products, {len(f8_product_week):,} product-weeks "
      f"(weeks 1-53, each judged against its own 13 earlier weeks)")
print(f8_product_week["flag"].value_counts().to_string())
print(f"\nProducts with at least one spike: {f8_flags.loc[f8_flags['flag']=='spike','unique_id'].nunique()} | "
      f"with at least one drop: {f8_flags.loc[f8_flags['flag']=='drop','unique_id'].nunique()}")

f8_by_week = (f8_flags.pivot_table(index=["week_no", "ds"], columns="flag", values="unique_id", aggfunc="count", fill_value=0)
              .reset_index().merge(f6_week_flags[["week_start", "holiday_names", "flag_total"]]
                                   .rename(columns={"week_start": "ds", "flag_total": "company_week"}), on="ds", how="left"))
for f8_c in ["spike", "drop"]:
    f8_by_week[f8_c] = f8_by_week[f8_c] if f8_c in f8_by_week.columns else 0
print("\nThe 8 weeks with the MOST products spiking (many products at once = a market-wide week, e.g. a season):")
display(f8_by_week.sort_values("spike", ascending=False).head(8)[["week_no", "ds", "spike", "drop", "holiday_names", "company_week"]])
print("\nThe 8 weeks with the MOST products dropping:")
display(f8_by_week.sort_values("drop", ascending=False).head(8)[["week_no", "ds", "spike", "drop", "holiday_names", "company_week"]])

f8_sp = f8_flags[f8_flags["flag"] == "spike"]
print(f"\nSpikes: {len(f8_sp)} | in holiday weeks: {int(f8_sp['is_holiday_week'].sum())} | "
      f"in after-holiday weeks: {int(f8_sp['after_holiday_week'].sum())}")
print("Top 15 spikes (x_usual = that week / the product's usual selling week):")
display(f8_sp.sort_values("x_usual", ascending=False).head(15)
        [["SKU", "ds", "week_no", "y", "usual_selling_week", "x_usual", "is_holiday_week", "family_backtest", "known_spike_note"]])

f8_dr = f8_flags[f8_flags["flag"] == "drop"]
print(f"\nDrops (regular sellers far below usual - possible stock-outs, ask EABL): {len(f8_dr)} product-weeks, "
      f"{f8_dr['unique_id'].nunique()} products | at zero: {int((f8_dr['y'] == 0).sum())}")
display(f8_dr.groupby(["SKU", "label_backtest"]).agg(drop_weeks=("ds", "count"), first=("ds", "min"), last=("ds", "max"),
                                                     usual_week=("usual_week", "median"))
        .sort_values("drop_weeks", ascending=False).head(15))

print("\nCleaning's Sunday spikes (known_spikes.csv, week of 8 Dec 2025) - did F8 flag them?")
display(f8_product_week[f8_product_week["known_spike_note"].notna()]
        [["Product_Code", "SKU", "y", "usual_selling_week", "x_usual", "flag", "known_spike_note"]])


In [ ]:
# ==========================================================================
# STEP 6 - PART 3 (F9): SAVE the week and product-week flags to ONE NEW file for F1
#          ../eabl parsed csv/product_week_flags.csv   (one row per product-week, weeks 1-53)
#
# WHY:
#   F6/F7 (Part 1) and F8 (Part 2) live only in this notebook's memory. F1 is a separate
#   notebook and cannot see them, so they must be saved to a file F1 can read (Step 7), where
#   they can become holiday / spike dummy variables (FPP2 5.4) instead of deleting weeks.
#   FPP2 4.2 "Document and justify": "Formalising and documenting the decision rules and
#   assumptions implemented in the systematic approach can promote consistency, as the same
#   rules can be implemented repeatedly."  -> one documented file, the same for every F1 run.
#   FPP2 12.9: replacing outliers "without thinking about why they have occurred is a dangerous
#   practice" -> the file holds FLAGS only; F1 decides (with EABL's answers) what to do with them.
#
# LEAK RULES (STEP 0 + Step 6):
#   - weeks 1-53 only (up to the seal, Mon 6 Apr 2026); no row from weeks 54-66 is saved
#   - every flag was made from EARLIER weeks only (F6: 8 earlier normal weeks; F8: 13 earlier
#     weeks), so a week's flag is the same for CV round 1 (<= wk 27), round 2 (<= wk 40) and
#     the test (<= wk 53). F1 may use any row whose ds is inside the round's learning weeks.
#   - NOT saved on purpose (to stop them being used by mistake):
#       label_backtest / class_backtest / family_backtest -> made from weeks 1-53; the right
#           labels per CV round are already in product_routing_labels.csv (F4)
#       known_spike_note -> from Cleaning's old all-dates rule (leak C); comparison only
#   - weeks 54-66 flags are NOT made here: they are only needed for the final refit (Step 9)
#
# WHERE EACH INPUT COMES FROM (all already in memory):
#   f8_product_week -> Step 6 Part 2 (F8, Prep cell 66): every product-week with its flag,
#                      usual_week, usual_selling_week, x_usual, week_no, holiday markers
#   f6_week_flags   -> Step 6 Part 1 (F6+F7, Prep cell 65): holiday_names, near_empty_days per week
#   F2B_CUTOFF      -> Fix 1 (Prep cell 29): 2026-04-06, the seal
#   pd, os          -> pandas (Prep cell 0); os is Python's built-in file helper
#
# WHAT IT CHANGES: nothing in memory - f8_product_week and f6_week_flags are only read.
# WHAT IT WRITES:  ONE NEW file, ../eabl parsed csv/product_week_flags.csv.
#                  It does NOT touch Y_product.csv, known_spikes.csv, product_routing_labels.csv
#                  or any other file. If a file with this name exists and was NOT made by this
#                  cell (no "made_by" = F9 column), the cell stops and writes nothing.
# All new names start with f9_ / F9_ so nothing clashes with other cells.
# ==========================================================================
import os

F9_PATH = "../eabl parsed csv/product_week_flags.csv"
F9_MADE_BY = "F9 Prep notebook (Step 6 Part 3)"

# ---- Safety checks: stop (and write nothing) if the inputs are not the expected tables ----
assert "f8_product_week" in globals() and len(f8_product_week) == 8930 and f8_product_week["unique_id"].nunique() == 181, \
    "run Step 6 Part 2 (F8, Prep cell 66) first"
assert "f6_week_flags" in globals() and len(f6_week_flags) == 53, "run Step 6 Part 1 (Prep cell 65) first"
assert f8_product_week["ds"].max() <= F2B_CUTOFF, "a week after the seal is in F8 - stop"

# ---- Build the table to save: F8 flag columns + F6 week columns, leak-risky columns left out ----
f9_out = (f8_product_week[["unique_id", "Product_Code", "SKU", "ds", "week_no", "y",
                           "usual_week", "usual_selling_week", "x_usual", "flag",
                           "is_holiday_week", "after_holiday_week", "company_week_flag"]]
          .merge(f6_week_flags[["week_start", "holiday_names", "near_empty_days"]]
                 .rename(columns={"week_start": "ds"}), on="ds", how="left")
          .rename(columns={"flag": "product_flag"})
          .sort_values(["unique_id", "ds"]).reset_index(drop=True))
f9_out["ds"] = f9_out["ds"].dt.date                       # plain date in the file (e.g. 2025-12-22)
f9_out["week_no"] = f9_out["week_no"].astype(int)
f9_out["cutoff"] = F2B_CUTOFF.date()                      # which weeks the flags could see
f9_out["made_by"] = F9_MADE_BY

# ---- Checks before writing ----
assert len(f9_out) == len(f8_product_week), "rows lost - stop and check"
assert not f9_out.duplicated(["unique_id", "ds"]).any(), "a product-week appears twice - stop and check"
assert f9_out["week_no"].between(1, 53).all() and max(f9_out["ds"]) <= F2B_CUTOFF.date()
assert abs(f9_out["y"].sum() - f8_product_week["y"].sum()) < 1, "cases changed - stop and check"
assert f9_out["product_flag"].isin(["spike", "drop", "normal", "too early"]).all()
assert not {"label_backtest", "class_backtest", "family_backtest", "known_spike_note"} & set(f9_out.columns)

# ---- Never overwrite a file this cell did not make ----
if os.path.exists(F9_PATH):
    f9_old = pd.read_csv(F9_PATH, nrows=1)
    assert "made_by" in f9_old.columns and str(f9_old["made_by"].iloc[0]).startswith("F9"), \
        f"{F9_PATH} already exists and was NOT made by F9. Nothing written; rename or check it."
    print(f"Note: {F9_PATH} already existed (made by an earlier F9 run) - it is replaced with this run's version.")

# ---- Write, then read back and compare (proves the file holds exactly the table) ----
f9_out.to_csv(F9_PATH, index=False)
f9_back = pd.read_csv(F9_PATH, dtype=str, keep_default_na=False)
assert f9_back.shape == f9_out.shape and (f9_back == f9_out.astype(str).replace("nan", "")).all().all(), \
    "the saved file does not match the table - stop and check"

# ---- Report ----
print(f"Saved {F9_PATH}: {f9_out.shape[0]:,} product-weeks x {f9_out.shape[1]} columns  (read back and checked - OK)")
print(f"Products: {f9_out['unique_id'].nunique()} | weeks: {f9_out['ds'].min()} to {f9_out['ds'].max()} (weeks 1-53)")
print("\nColumns:", list(f9_out.columns))
print("\nProduct flags:", f9_out["product_flag"].value_counts().to_dict())
print("Company-week flags (product-weeks):", f9_out["company_week_flag"].value_counts().to_dict())
print(f"Product-weeks in holiday weeks: {int(f9_out['is_holiday_week'].sum()):,} | "
      f"in after-holiday weeks: {int(f9_out['after_holiday_week'].sum()):,}")
print("\nExample rows (Don Julio Blanco around Christmas 2025):")
display(f9_out[f9_out["SKU"].str.startswith("Don Julio Blanco") & f9_out["week_no"].between(36, 39)]
        [["SKU", "ds", "week_no", "y", "usual_selling_week", "x_usual", "product_flag",
          "is_holiday_week", "holiday_names", "company_week_flag"]])


In [ ]:

# Only Product_Description is genuinely missing from your working dataset —
# pull just that one column from the master file
pm = pd.read_csv("../eabl parsed csv/product_master_cleaned.csv")[["Product_Code", "Product_Description"]]
pm["Product_Code"] = pm["Product_Code"].astype(str)

# Brand and Category2 already exist in your working dataset — grab them from there
product_lookup = df[["Product_Code", "Brand", "Category2"]].drop_duplicates()
product_lookup["Product_Code"] = product_lookup["Product_Code"].astype(str)
product_lookup = product_lookup.merge(pm, on="Product_Code", how="left")

classification_product["unique_id"] = classification_product["unique_id"].astype(str)
classification_named = classification_product.merge(
    product_lookup, left_on="unique_id", right_on="Product_Code", how="left"
)

for group_name in ["smooth", "intermittent", "erratic", "lumpy"]:
    subset = classification_named[classification_named["class"] == group_name]
    print(f"\n=== {group_name.upper()} — {len(subset)} products ===")
    print(subset[["Product_Code", "Product_Description", "Brand", "Category2"]].to_string(index=False))



In [ ]:
routing_table = pd.concat([
    classification_product.assign(hierarchy="Product"),
    geo_check.assign(hierarchy="Geographic"),
    channel_check.assign(hierarchy="Channel"),
    grouped_check.assign(hierarchy="Grouped"),
])

routing_table["model_family"] = routing_table["class"].map({
    "smooth": "SeasonalNaive/AutoETS",
    "erratic": "SeasonalNaive/AutoETS",     # still worth standard methods, just noisier
    "intermittent": "Croston/SBA/TSB",
    "lumpy": "Croston/SBA/TSB",
})

print(routing_table.groupby(["hierarchy", "model_family"]).size())


## Forecasting Playbook — Hierarchy Priority & Method Routing

## The Four Hierarchies

| Letter | Name | Structure |
|---|---|---|
| **A** | **Product** | Total → Category2 → Sub_Category → Brand → Product_Code (SKU) |
| **B** | **Geographic** | Total → Division → Area → Territory → Route (using Territory+Route composite key) |
| **C** | **Channel** | Total → global Channel → Sub-Channel → segment_grouped (Bar, Nightclub, Wholesaler, Specialist Store, Event, Lodging, Other) |
| **D** | **Grouped cross-cut** | Category2 × segment_grouped (e.g. "how much Spirits sold specifically through Nightclubs") |


**Decided on:** 2026-09-28, based on ADI/CV² intermittency classification (Step 4.4)

**Numbers updated:** 2026-11-04, after F5 (classification starts at each product's launch).
**Scope:** only **A — Product** is forecast in this project. B, C and D are kept for reference only.


### Priority order to forecast (easiest/most reliable first)
1. **D — Grouped (Category × Segment):** 86% smooth, 0% lumpy/intermittent.
   → Use a single method: SeasonalNaive + AutoETS for the whole hierarchy.
2. **C — Channel:** 59% smooth (small sample, 17 nodes — treat cautiously).
   → Mostly SeasonalNaive/AutoETS; manually check the 7 non-smooth nodes.
3. **B — Geographic (Route):** 60% smooth (25 of 42) — needs a split approach.

4. **A — Product (SKU):** only 44% smooth (82 of 188; 82 smooth / 41 erratic / 50 lumpy / 15 intermittent) — needs the most care.
   
### Why this order
Start with the hierarchy needing the least special handling (D) to confirm
the pipeline works end-to-end, before tackling the two hierarchies (A, B)
that need a split-method approach.

### After all four have base forecasts
Reconcile using the S matrices already built in Step 2.3, so Product,
Geographic, Channel, and Total numbers all agree with each other.

### Reference
### Reference
For the **Product** hierarchy, F1 must use `../eabl parsed csv/product_routing_labels.csv`
(F4, Prep cell 65): it holds the class and family for each set (cv1, cv2, backtest, final),
each made only from that set's own weeks. `routing_table.csv` is the older, all-weeks version
and must not be used for choosing models.


In [ ]:
routing_table = pd.concat([
    classification_product.assign(hierarchy="Product"),
    geo_check.assign(hierarchy="Geographic"),
    channel_check.assign(hierarchy="Channel"),
    grouped_check.assign(hierarchy="Grouped"),
])

routing_table["model_family"] = routing_table["class"].map({
    "smooth": "SeasonalNaive/AutoETS",
    "erratic": "SeasonalNaive/AutoETS",
    "intermittent": "Croston/SBA/TSB",
    "lumpy": "Croston/SBA/TSB",
})

routing_table.to_csv("../eabl parsed csv/routing_table.csv", index=False)
print("Saved:", routing_table.shape)


In [ ]:

# Reuse the true company-wide series we already built for the 3.7 structural-break check
total_by_week = Y_product[Y_product["unique_id"].isin(["Beer", "RTD", "Spirits", "Returnable"])]
print(total_by_week)
total_by_week = total_by_week.groupby("ds")["y"].sum().reset_index().sort_values("ds")
total_series = total_by_week.set_index("ds")["y"]

stl = STL(total_series, period=52, robust=True).fit()
seasonal_strength = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
print("Seasonal strength (0 = none, 1 = very strong):", seasonal_strength)

stl.plot()


In [ ]:
def get_company_total(Y_df, tags, top_level_spec):
    """
    Sums all top-level nodes together to get the true company-wide total
    for any hierarchy, using tags instead of guessing label names.
    """
    top_level_tag = "/".join(top_level_spec)
    top_level_ids = tags[top_level_tag]
    total = (
        Y_df[Y_df["unique_id"].isin(top_level_ids)]
        .groupby("ds")["y"].sum()
        .reset_index()
        .sort_values("ds")
    )
    return total.set_index("ds")["y"]


# Now this works safely for all four, no guessing involved
total_product = get_company_total(Y_product, tags_product, spec_product[0])
total_geo = get_company_total(Y_geo, tags_geo, spec_geo[0])
total_channel = get_company_total(Y_channel, tags_channel, spec_channel[0])
total_grouped = get_company_total(Y_grouped, tags_grouped, spec_grouped[0])


In [ ]:

def check_seasonality(series, name):
    stl = STL(series, period=52, robust=True).fit()
    strength = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
    print(f"{name} seasonal strength: {strength:.3f}")
    return strength

check_seasonality(total_product, "Product (A)")
check_seasonality(total_geo, "Geographic (B)")
check_seasonality(total_channel, "Channel (C)")
check_seasonality(total_grouped, "Grouped (D)")


In [ ]:

def check_and_plot_seasonality(series, name):
    stl = STL(series, period=52, robust=True).fit()
    strength = max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var())
    print(f"{name} seasonal strength: {strength:.3f}")

    fig = stl.plot()
    fig.suptitle(f"{name} — Trend / Seasonal / Residual decomposition", y=1.02)
    fig.set_size_inches(10, 8)
    plt.tight_layout()
    plt.show()

    return strength





In [ ]:
strengths = {}
strengths["Product (A)"] = check_and_plot_seasonality(total_product, "Product (A)")


In [ ]:
strengths["Geographic (B)"] = check_and_plot_seasonality(total_geo, "Geographic (B)")


In [ ]:
strengths["Channel (C)"] = check_and_plot_seasonality(total_channel, "Channel (C)")


In [ ]:
strengths["Grouped (D)"] = check_and_plot_seasonality(total_grouped, "Grouped (D)")

print("\nSummary of seasonal strength across hierarchies:")
for name, val in strengths.items():
    print(f"  {name}: {val:.3f}")

In [ ]:
comparison = total_product.rename("product").to_frame()
comparison["geographic"] = total_geo
comparison["channel"] = total_channel
comparison["grouped"] = total_grouped

comparison["all_match"] = (
    (comparison["product"] == comparison["geographic"]) &
    (comparison["product"] == comparison["channel"]) &
    (comparison["product"] == comparison["grouped"])
)

print("Weeks where all four totals match exactly:", comparison["all_match"].sum(), "/", len(comparison))
print(comparison[~comparison["all_match"]].head(10))


In [ ]:

comparison["all_match"] = (
    np.isclose(comparison["product"], comparison["geographic"], rtol=1e-6) &
    np.isclose(comparison["product"], comparison["channel"], rtol=1e-6) &
    np.isclose(comparison["product"], comparison["grouped"], rtol=1e-6)
)

print("Weeks where all four totals match (within tiny rounding tolerance):", comparison["all_match"].sum(), "/", len(comparison))


In [ ]:
# ==========================================================================
# SCORECARD ROW 4: IS THERE A REAL 13-WEEK SEASON?  (exploratory analysis on weeks 1-53 only)
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   1.6 Step 3 (p.19): "Always start by graphing the data. ... Is seasonality important?"
#   2.3 (p.32): "Seasonality is always of a fixed and known frequency."
#   2.2 time plot, 2.4 seasonal plot (p.34), 2.8 ACF (p.48): "When data are seasonal, the
#     autocorrelations will be larger for the seasonal lags (at multiples of the seasonal
#     frequency) than for other lags."
#   2.9 (p.51): for a series with no pattern, 95% of ACF spikes lie within +-2/sqrt(T).
#   12.1 (p.462): weekly data - the yearly period (52.18 weeks) is large and non-integer.
#   12.7 (p.479): "we need more observations than there are parameters in our forecasting model."
#   F1 uses season_length=13 in 12 places (F1 cells 67, 68, 104) - this cell tests that choice.
#
# THE TEST (rule written BEFORE looking at the result):
#   A 13-week season is supported only if BOTH:
#     (a) the company Total has r13 above the +-2/sqrt(T) bound AND r13 is larger than its
#         neighbours r12 and r14 (FPP2 2.8: seasonal lags stand out from the other lags), and
#     (b) clearly more products than chance have a significant positive r13: FPP2 2.9 expects
#         5% by chance; "clearly more" is set here at 10% (twice chance).
#   Otherwise: season_length=13 is NOT supported by the data.
#   A yearly season (52 weeks) cannot be tested: weeks 1-53 hold only ONE year, and the ACF at
#   lag 52 would rest on a single pair of weeks.
#
# LEAK RULES: weeks 1-53 only (ds <= F2B_CUTOFF, the seal). The test weeks 54-66 are not used,
#   so this decision can be applied to the CV rounds and the backtest without peeking.
#
# WHERE EACH INPUT COMES FROM (already in memory):
#   Y_product     -> Prep cell 23 (STEP 8 zero-fill): columns unique_id / ds / y, every level of the
#                    Product hierarchy, weekly cases, 66 Monday weeks (also saved as Y_product.csv)
#   tags_product  -> Prep cell 22: dict level name -> list of unique_ids at that level
#   F2B_CUTOFF    -> Fix 1 (Prep cell 29): Monday 6 Apr 2026, the seal (week 53)
#   pd, np, plt   -> Prep cell 0 (imports);  display -> Jupyter built-in
#   acf           -> NEW import below, from statsmodels (already installed: cell 0 imports STL from it)
#
# WHAT IT CHANGES: nothing - Y_product and tags_product are only read.
# WHAT IT WRITES:  nothing (no file is saved). It shows 3 plots and 2 tables.
# All new names start with eda4_ / EDA4_.
# ==========================================================================
from statsmodels.tsa.stattools import acf

EDA4_LEVELS = {"Category2": "Category2", "Sub_Category": "Category2/Sub_Category",
               "Brand": "Category2/Sub_Category/Brand", "Product": "Category2/Sub_Category/Brand/Product_Code"}
EDA4_MAX_LAG = 26           # two 13-week cycles
EDA4_CHANCE = 0.05          # FPP2 2.9: 5% of spikes outside the bounds by chance
EDA4_CLEARLY_MORE = 0.10    # "clearly more than chance" = twice chance (my choice, stated before the result)

# ---- Safety checks ----
assert {"unique_id", "ds", "y"}.issubset(Y_product.columns), "Y_product missing - run Prep cell 23"
assert set(EDA4_LEVELS.values()).issubset(tags_product.keys()), "tags_product missing - run Prep cell 22"
assert F2B_CUTOFF == pd.Timestamp("2026-04-06"), "run Fix 1 (Prep cell 29) first"

# ---- Weeks 1-53 only ----
eda4_Y = Y_product[Y_product["ds"] <= F2B_CUTOFF]
eda4_weeks = pd.Index(sorted(eda4_Y["ds"].unique()))
assert len(eda4_weeks) == 53 and eda4_weeks.max() == F2B_CUTOFF, "expected 53 weeks up to the seal"

# Company Total = sum of the Category2 nodes (the hierarchy has no Total row yet - scorecard row 3)
eda4_total = (eda4_Y[eda4_Y["unique_id"].isin(tags_product["Category2"])]
              .groupby("ds")["y"].sum().reindex(eda4_weeks))
assert eda4_total.notna().all()
eda4_T = len(eda4_total)
eda4_bound = 2 / np.sqrt(eda4_T)
eda4_r = acf(eda4_total.values, nlags=EDA4_MAX_LAG, fft=False)          # eda4_r[k] = autocorrelation at lag k
eda4_rd = acf(eda4_total.diff().dropna().values, nlags=EDA4_MAX_LAG, fft=False)   # after removing the trend
eda4_bound_d = 2 / np.sqrt(eda4_T - 1)

# ---- Plot 1 (FPP2 2.2): time plot ----
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(eda4_total.index, eda4_total.values, marker="o", ms=3)
ax.set_title(f"Time plot - company total, weeks 1-53 ({eda4_weeks[0].date()} to {eda4_weeks[-1].date()})", loc="left")
ax.set_ylabel("cases per week")
plt.show()

# ---- Plot 2 (FPP2 2.4): seasonal plot, one line per 13-week cycle ----
eda4_week_no = np.arange(1, eda4_T + 1)
eda4_season = pd.DataFrame({"y": eda4_total.values, "cycle": (eda4_week_no - 1) // 13 + 1,
                            "week_in_cycle": (eda4_week_no - 1) % 13 + 1})
fig, ax = plt.subplots(figsize=(8, 4))
for eda4_c, eda4_g in eda4_season[eda4_season["cycle"] <= 4].groupby("cycle"):   # 4 full cycles = weeks 1-52
    ax.plot(eda4_g["week_in_cycle"], eda4_g["y"], marker="o", ms=3,
            label=f"cycle {eda4_c} (weeks {13 * (eda4_c - 1) + 1}-{13 * eda4_c})")
ax.set_xticks(range(1, 14)); ax.set_xlabel("week inside the 13-week cycle"); ax.set_ylabel("cases per week")
ax.set_title("Seasonal plot - if a 13-week season were real, the 4 lines would have the same shape", loc="left")
ax.legend(fontsize=8)
plt.show()

# ---- Plot 3 (FPP2 2.8 / 2.9): ACF with the +-2/sqrt(T) bounds ----
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, rr, bb, name in [(axes[0], eda4_r, eda4_bound, "weekly total"),
                         (axes[1], eda4_rd, eda4_bound_d, "week-to-week change (trend removed)")]:
    ax.bar(range(1, EDA4_MAX_LAG + 1), rr[1:], width=0.4)
    ax.axhline(bb, ls="--", c="tab:blue"); ax.axhline(-bb, ls="--", c="tab:blue"); ax.axhline(0, c="grey", lw=0.8)
    for k in (13, 26):
        ax.axvline(k, c="tab:red", lw=0.8, alpha=0.5)
    ax.set_title(f"ACF - {name}", loc="left"); ax.set_xlabel("lag (weeks); red lines = 13 and 26")
plt.show()

# ---- Every level of the Product hierarchy: how many series have a significant r13? ----
eda4_rows = []
for eda4_level, eda4_key in EDA4_LEVELS.items():
    for eda4_id in tags_product[eda4_key]:
        eda4_s = eda4_Y.loc[eda4_Y["unique_id"] == eda4_id].set_index("ds")["y"]
        if len(eda4_s) < eda4_T or eda4_s.std() == 0:
            continue                       # only series with all 53 weeks (same T, same bound) and some variation
        eda4_rr = acf(eda4_s.reindex(eda4_weeks).values, nlags=EDA4_MAX_LAG, fft=False)
        eda4_rows.append({"level": eda4_level, "unique_id": eda4_id, "r13": eda4_rr[13],
                          "r13_significant": eda4_rr[13] > eda4_bound,
                          "share_lags_outside": np.mean(np.abs(eda4_rr[1:]) > eda4_bound)})
eda4_levels = pd.DataFrame(eda4_rows)
eda4_summary = (eda4_levels.groupby("level", sort=False)
                .agg(series_with_53_weeks=("unique_id", "size"),
                     median_r13=("r13", "median"),
                     share_r13_significant=("r13_significant", "mean"),
                     share_all_lags_outside=("share_lags_outside", "mean"))
                .round(3))
eda4_summary.loc["Total"] = [1, round(eda4_r[13], 3), float(eda4_r[13] > eda4_bound),
                             round(np.mean(np.abs(eda4_r[1:]) > eda4_bound), 3)]
eda4_summary = eda4_summary.loc[["Total"] + list(EDA4_LEVELS)]

# ---- The decision (rule written at the top) ----
eda4_a = (eda4_r[13] > eda4_bound) and (eda4_r[13] > eda4_r[12]) and (eda4_r[13] > eda4_r[14])
eda4_prod_share = eda4_summary.loc["Product", "share_r13_significant"]
eda4_b = eda4_prod_share > EDA4_CLEARLY_MORE
EDA4_SEASON_13_SUPPORTED = bool(eda4_a and eda4_b)

print(f"Weeks used: 1-53 ({eda4_weeks[0].date()} to {eda4_weeks[-1].date()}), T = {eda4_T}, "
      f"bound +-2/sqrt(T) = +-{eda4_bound:.3f}  (FPP2 2.9)\n")
print("Company total ACF:")
print(f"  r12 = {eda4_r[12]:.2f} | r13 = {eda4_r[13]:.2f} | r14 = {eda4_r[14]:.2f} | r26 = {eda4_r[26]:.2f}")
print(f"  lags outside the bounds: {[k for k in range(1, EDA4_MAX_LAG + 1) if abs(eda4_r[k]) > eda4_bound]}")
print(f"  trend removed: r13 = {eda4_rd[13]:.2f}, lags outside: "
      f"{[k for k in range(1, EDA4_MAX_LAG + 1) if abs(eda4_rd[k]) > eda4_bound_d]}\n")
print("Every level (series with all 53 weeks):")
display(eda4_summary)
print(f"\n(a) Total r13 significant and larger than r12, r14?  {eda4_a}")
print(f"(b) Products with a significant r13: {eda4_prod_share:.1%}  (chance alone gives about "
      f"{EDA4_CHANCE:.0%}; needed > {EDA4_CLEARLY_MORE:.0%})  ->  {eda4_b}")
print(f"\nDECISION: 13-week season supported by weeks 1-53?  {EDA4_SEASON_13_SUPPORTED}")
print("Yearly season (52 weeks): cannot be tested - weeks 1-53 contain only one year (FPP2 12.1, 12.7).")


In [ ]:
# ==========================================================================
# SCORECARD ROW 3: ADD THE TOTAL (LEVEL 0) ROW TO THE PRODUCT HIERARCHY
#          -> saved as THREE NEW files; Y_product.csv and S_product.csv are NOT touched
#
# WHY (FPP2 2nd ed., Hyndman & Athanasopoulos 2018):
#   10.1 (p.394): "At the top of the hierarchy (which we call level 0) is the "Total", the most
#     aggregate level of the data."
#   10.1 (p.395): "the first row in the summing matrix S represents Equation (10.1)" - i.e. the
#     Total row of S is all 1s: Total = the sum of every bottom-level product.
#   10.2 (p.409): top-down approaches "involve first generating forecasts for the Total series,
#     and then disaggregating these down the hierarchy" - impossible without a Total series.
#   Prep cell 22 built the hierarchy with spec_product starting at Category2, so the top of the
#   tree was 3 separate series (Beer, RTD, Spirits) and no single Total.
#
# HOW: one new series "Total" is ADDED on top. Every existing unique_id stays exactly the same
#   (e.g. "Beer/ANADS/Alvaro/672279"), so product_routing_labels.csv and product_week_flags.csv
#   still match row for row. Nothing is re-aggregated from the raw data.
#     Y:    Total y in each week = sum of the 188 products' y in that week
#     S:    one new first row "Total" = 1 under every product column
#     tags: new first level "Total" = ["Total"]; the 4 existing levels unchanged
#
# LEAK RULES: the Total of a week uses only that same week's sales (a sum across products, never
#   across weeks), so it adds no information from later weeks. All 66 weeks are kept, exactly like
#   Y_product.csv; F1 cuts each round at its own cut-off (Step 7), as for every other series.
#
# WHERE EACH INPUT COMES FROM (already in memory):
#   Y_product, S_product, tags_product -> Prep cell 23 (rebuilt from the zero-filled weeks with
#                     aggregate(df=long_product, spec=spec_product)); same data as Y_product.csv /
#                     S_product.csv saved by Prep cell 25
#   spec_product   -> Prep cell 22: the 4 levels Category2 -> ... -> Product_Code
#   pd, np, os     -> pandas / numpy (Prep cell 0); os is Python's built-in file helper
#   json           -> Python's built-in JSON helper (to save tags)
#
# WHAT IT CHANGES IN MEMORY: nothing existing. It creates Y_product_total, S_product_total and
#   tags_product_total; Y_product, S_product and tags_product are only read.
# WHAT IT WRITES: three NEW files in ../eabl parsed csv/
#   Y_product_with_total.csv      (unique_id, ds, y)             283 series
#   S_product_with_total.csv      (same layout as S_product.csv) 283 rows x 188 product columns
#   tags_product_with_total.json  (level name -> list of unique_ids)
#   If any of these names already exists and was NOT made by this cell, the cell stops and
#   writes nothing.
# All new names start with tot3_ / TOT3_ (plus the three *_total tables).
# ==========================================================================
import os
import json

TOT3_DIR = "../eabl parsed csv/"
TOT3_Y_PATH = TOT3_DIR + "Y_product_with_total.csv"
TOT3_S_PATH = TOT3_DIR + "S_product_with_total.csv"
TOT3_TAGS_PATH = TOT3_DIR + "tags_product_with_total.json"
TOT3_BOTTOM = "/".join(spec_product[-1])          # "Category2/Sub_Category/Brand/Product_Code"

# ---- Safety checks on the inputs ----
assert {"unique_id", "ds", "y"}.issubset(Y_product.columns), "Y_product missing - run Prep cell 23"
assert "unique_id" in S_product.columns and len(S_product) == Y_product["unique_id"].nunique() == 282, \
    "S_product / Y_product are not the 282-series Product hierarchy - run Prep cells 22-23"
assert list(tags_product.keys()) == ["/".join(lvl) for lvl in spec_product], "tags_product missing - run Prep cell 23"
assert "Total" not in set(Y_product["unique_id"]), "Y_product already has a Total row - stop and check"
tot3_bottom_ids = list(tags_product[TOT3_BOTTOM])
assert len(tot3_bottom_ids) == 188

# ---- Y: one Total value per week = sum of the 188 products in that week ----
tot3_y = (Y_product[Y_product["unique_id"].isin(tot3_bottom_ids)]
          .groupby("ds", as_index=False)["y"].sum())
tot3_y.insert(0, "unique_id", "Total")
Y_product_total = pd.concat([tot3_y, Y_product], ignore_index=True)[["unique_id", "ds", "y"]]

# ---- S: one new first row of 1s (every product belongs to the Total) ----
tot3_S_row = pd.DataFrame([[ "Total"] + [1.0] * len(tot3_bottom_ids)], columns=["unique_id"] + tot3_bottom_ids)
S_product_total = pd.concat([tot3_S_row, S_product], ignore_index=True)[["unique_id"] + tot3_bottom_ids]

# ---- tags: new first level "Total"; the 4 existing levels unchanged ----
tags_product_total = {"Total": np.array(["Total"], dtype=object)} | {k: v for k, v in tags_product.items()}

# ---- PROOF 1 (FPP2 10.1): the Total equals Beer + RTD + Spirits in every week ----
tot3_cat = (Y_product[Y_product["unique_id"].isin(tags_product["Category2"])]
            .groupby("ds")["y"].sum())
tot3_tot = tot3_y.set_index("ds")["y"]
assert tot3_tot.index.equals(tot3_cat.index) and np.allclose(tot3_tot, tot3_cat), \
    "Total is not equal to the sum of the 3 categories - stop and check"

# ---- PROOF 2 (FPP2 10.1): y = S b in EVERY one of the 66 weeks, for all 283 series ----
tot3_wide = Y_product_total.pivot(index="unique_id", columns="ds", values="y").fillna(0.0)
tot3_S = S_product_total.set_index("unique_id")
tot3_b = tot3_wide.loc[tot3_bottom_ids]                                   # 188 products x 66 weeks
tot3_rebuilt = tot3_S.values @ tot3_b.values                              # 283 series x 66 weeks
tot3_actual = tot3_wide.reindex(tot3_S.index).values
assert np.allclose(tot3_rebuilt, tot3_actual), "y = S b does not hold - stop and check"
tot3_weeks = tot3_wide.shape[1]

# ---- PROOF 3: nothing existing changed ----
assert Y_product_total[Y_product_total["unique_id"] != "Total"].reset_index(drop=True) \
       .equals(Y_product[["unique_id", "ds", "y"]].reset_index(drop=True)), "existing Y rows changed - stop"
assert S_product_total.iloc[1:].reset_index(drop=True) \
       .equals(S_product[["unique_id"] + tot3_bottom_ids].reset_index(drop=True)), "existing S rows changed - stop"

# ---- Never overwrite a file this cell did not make ----
def tot3_made_here(path):
    """True if path does not exist, or it exists and was written by this cell before."""
    if not os.path.exists(path):
        return True
    if path.endswith(".json"):
        with open(path) as f:
            return json.load(f).get("_made_by", "").startswith("Scorecard row 3")
    return pd.read_csv(path, nrows=1).iloc[0].get("unique_id") == "Total"
for tot3_path in (TOT3_Y_PATH, TOT3_S_PATH, TOT3_TAGS_PATH):
    assert tot3_made_here(tot3_path), f"{tot3_path} exists and was NOT made by this cell. Nothing written."

# ---- Write, then read back and compare ----
Y_product_total.to_csv(TOT3_Y_PATH, index=False)
S_product_total.to_csv(TOT3_S_PATH)                   # same layout as S_product.csv (index + unique_id + products)
with open(TOT3_TAGS_PATH, "w") as f:
    json.dump({"_made_by": "Scorecard row 3 (Prep notebook): Product hierarchy with Total",
               **{k: [str(x) for x in v] for k, v in tags_product_total.items()}}, f, indent=1)

tot3_Y_back = pd.read_csv(TOT3_Y_PATH, parse_dates=["ds"])
tot3_S_back = pd.read_csv(TOT3_S_PATH, index_col=0)
with open(TOT3_TAGS_PATH) as f:
    tot3_tags_back = json.load(f)
assert tot3_Y_back.shape == Y_product_total.shape and np.allclose(tot3_Y_back["y"], Y_product_total["y"])
assert (tot3_Y_back["unique_id"] == Y_product_total["unique_id"]).all()
assert tot3_S_back.shape == S_product_total.shape and np.allclose(tot3_S_back[tot3_bottom_ids].values,
                                                                  S_product_total[tot3_bottom_ids].values)
assert [k for k in tot3_tags_back if k != "_made_by"] == list(tags_product_total)

# ---- Report ----
print("Scorecard row 3 - Total (level 0) added to the Product hierarchy  (FPP2 10.1)\n")
print(f"Series: {Y_product['unique_id'].nunique()} -> {Y_product_total['unique_id'].nunique()}  "
      f"(+1 'Total')  |  S: {S_product[tot3_bottom_ids].shape} -> {S_product_total[tot3_bottom_ids].shape}")
print("Levels:", {k: len(v) for k, v in tags_product_total.items()})
print(f"Proof 1: Total = Beer + RTD + Spirits in all {len(tot3_tot)} weeks - OK")
print(f"Proof 2: y = S b holds for all {tot3_S.shape[0]} series in all {tot3_weeks} weeks - OK")
print("Proof 3: every existing Y and S row is unchanged - OK")
print(f"\nSaved (read back and checked - OK):\n  {TOT3_Y_PATH}  {Y_product_total.shape}\n"
      f"  {TOT3_S_PATH}  {S_product_total.shape}\n  {TOT3_TAGS_PATH}")
print("NOT touched: Y_product.csv, S_product.csv and every other file.")
print("\nTotal, first 5 weeks:")
display(tot3_y.head().assign(y=lambda d: d["y"].round(2)))
print(f"Total cases, all {len(tot3_tot)} weeks: {tot3_tot.sum():,.2f}  |  weeks 1-53: "
      f"{tot3_tot[tot3_tot.index <= pd.Timestamp('2026-04-06')].sum():,.2f}")